# '02.-Actualización Contable Marine' - PROGRAM 02

##  Overview
In this program, we use the bd_update.pkl processed in the previous program.
After that, load and prepare the Accounting information.
Cross the information of bd_update and the account information.
Finally, create the formulated variables and export.
This program creates the final database.

##  Execution Flow
1. Library Import.
2. Path Definition and Macrovariables. 
3. Data import (bd_update.pkl & Account Information). 
4. Data Format.
5. Data Cross.
6. Creation of formulated variables.
7. OSLR and Dates validation.
8. Final adjustments.
9. Final Export.

##  Output
- Excel file with the final processed database of the month. ({AñoMes}_Siniestros_Marine.xlsx)
- Pickle file with the final processed database of the month. ({AñoMes}_Siniestros_Marine.pkl)
- Excel file with relevant information for incidences analysis. (actuarial.xlsx)

## 1. Library import

In [170]:
## =============================================================================
# Section 1: Library import
# =============================================================================

# === Optional: Clean all the enviroment prior running === #
%reset -f                                                  
# ======================================================== #
import os
import pandas as pd
#import dtale
import numpy as np
import re
import timeit
import sqlite3

start_time = timeit.default_timer() # Timer

## 1.5 Importación de configuración centralizada

In [171]:
# =============================================================================
# Importación centralizada de configuración
# =============================================================================
import sys
sys.path.insert(0, r'C:/Users/IKAL14/Documents/Integral/Marine/Diccionarios')
from config_marine import *

## 2. Path Definition and Macrovariables 

In [172]:
## =============================================================================
# Section 2: Path Definition and Macrovariables. 
# =============================================================================
print('===============================================================================================')
print('Inicio del proceso')
start_time = timeit.default_timer() # Timer

# ================= Edit variables month================================ #
AñoMes = 202604
AñoMes_ant = 202512

# Convert the variables in datetime 
date_start_AñoMes = pd.to_datetime(str(AñoMes), format='%Y%m') + pd.offsets.MonthBegin(0)
date_start_AñoMes_ant = pd.to_datetime(str(AñoMes_ant), format='%Y%m') + pd.offsets.MonthBegin(0)
date_start_suma = date_start_AñoMes_ant + pd.offsets.MonthBegin(1) # This is an Aux Variable and it should be used when we are proccesing more than 1 month

print(f'Fecha de inicio para considerar en la suma: {date_start_suma}')

# =================Routes definition========================================= #
print('===============================================================================================')

# Define your project directory path as a variable
directorio_proyecto =  "C:/Users/IKAL14/Documents/Integral/Marine"   #C:/Users/KOT23/Documents/Integral

# Change the current working directory to the project directory
os.chdir(directorio_proyecto) 

# Verify that the current directory is the project directory
print(f"Directorio actual de trabajo: {os.getcwd()}")

# Processed files path
ruta_procesados = rf"{directorio_proyecto}/Procesados/{AñoMes}" 
print(f"Ruta de archivos procesados: {ruta_procesados}")

#Ruta de Bases
ruta_bases= rf"{directorio_proyecto}/Bases"
print(f"Ruta de bases: {ruta_bases}")

# Incidences files path
ruta_incidencias = rf"{directorio_proyecto}/Incidencias/{AñoMes}" 
print(f"Ruta de Incidencias: {ruta_incidencias}")

# Validation files path
route_validations = rf"{directorio_proyecto}/Validaciones/{AñoMes}" 
print(f"Ruta de Validaciones: {route_validations}")

# Catalog files path
route_catalog = rf'{directorio_proyecto}/Catalogos'
print(f'Ruta de catalogos: {route_catalog}')

# Actuarial database path
route_actuarial = rf'C:/Users/IKAL14/Documents/Integral/Insumos'
print(f'Ruta de base actuarial: {route_actuarial}')

# Payments information
route_account = fr'C:/Users/IKAL14/Documents/Integral/Insumos/Contabilidad/{AñoMes}'
print(f'Ruta de base de payments: {route_account}')

# 'Pruebas' Route
route_pruebas = rf"{directorio_proyecto}/Procesados/{AñoMes}/pruebas" 
# Verify if the path exists
if not os.path.exists(route_pruebas):
    os.makedirs(route_pruebas)
    print(f"Carpeta creada: {route_pruebas}")
else:
    print(f"La carpeta ya existe: {route_pruebas}")

# 'OSLR' Route
route_oslr = rf"{directorio_proyecto}/Procesados/202504" 

# This DB includes all of the records that can be updated
df_update_db = pd.read_pickle(f'{ruta_procesados}/bd_update.pkl')
df_update_db['CLAIM NUMBER'] = df_update_db['CLAIM NUMBER'].astype(str)
df_update_db['LoB-Inward'] = df_update_db['LoB-Inward'].astype(str)

Inicio del proceso
Fecha de inicio para considerar en la suma: 2026-01-01 00:00:00
Directorio actual de trabajo: C:\Users\IKAL14\Documents\Integral\Marine
Ruta de archivos procesados: C:/Users/IKAL14/Documents/Integral/Marine/Procesados/202604
Ruta de bases: C:/Users/IKAL14/Documents/Integral/Marine/Bases
Ruta de Incidencias: C:/Users/IKAL14/Documents/Integral/Marine/Incidencias/202604
Ruta de Validaciones: C:/Users/IKAL14/Documents/Integral/Marine/Validaciones/202604
Ruta de catalogos: C:/Users/IKAL14/Documents/Integral/Marine/Catalogos
Ruta de base actuarial: C:/Users/IKAL14/Documents/Integral/Insumos
Ruta de base de payments: C:/Users/IKAL14/Documents/Integral/Insumos/Contabilidad/202604
La carpeta ya existe: C:/Users/IKAL14/Documents/Integral/Marine/Procesados/202604/pruebas


## 3. Data import

In [173]:
# =============================================================================
# Section 3: Data import
# =============================================================================

# ======== Import of update database ======== #

# ========== FIX #1: Normalizar CLAIM NUMBER antes de crear KEY LOB ==========
# Elimina espacios internos para garantizar consistencia entre períodos.
df_update_db['CLAIM NUMBER'] = (
    df_update_db['CLAIM NUMBER']
    .str.strip()
    .str.replace(r'\s+', '', regex=True)
)

# ========== FIX #2: Normalizar LoB-Inward antes de crear KEY LOB ==========
# Unifica LoB que cambian de nombre entre períodos para mantener continuidad.
LOB_NORMALIZATION = {
    'CASCO': 'CASCO Y MAQ.',
}
df_update_db['LoB-Inward'] = (
    df_update_db['LoB-Inward']
    .str.strip()
    .replace(LOB_NORMALIZATION)
)

# Crear KEY LOB con formato 100% consistente
df_update_db['KEY LOB'] = df_update_db['CLAIM NUMBER']  + "-" +  df_update_db['LoB-Inward']

# Rename of the columns
df_update_db.rename(columns= {
                              'DEDUCTIBLE': f'DEDUCTIBLE {AñoMes_ant}' , 'DEDUCTIBLE BDX': f'DEDUCTIBLE {AñoMes}',
                              'GROSS RESERVE': f'GROSS RESERVE {AñoMes_ant}', 'GROSS RESERVE BDX': f'GROSS RESERVE {AñoMes}',
                              'Cumulative EXPENSES PAID':f'Cumulative EXPENSES PAID {AñoMes_ant}',
                              'Cumulative VALUATION EXPENSES':f'Cumulative VALUATION EXPENSES {AñoMes_ant}',
                              'Cumulative CLAIMS PAID':f'Cumulative CLAIMS PAID {AñoMes_ant}',
                              'OSLR Inward':f'OSLR Inward {AñoMes_ant}'}, inplace= True)

# ======== Import of account information ======== #
# Initialize a dictionary with the sheets of information
dict_sheets = pd.read_excel(fr'{route_account}/Payments_{AñoMes}.xlsx', sheet_name= None)
# Split the dictionary in dataframes
df_AE = dict_sheets['AE'] 
df_CL = dict_sheets['CL']
df_VA = dict_sheets['VA']

# Cleaning
del dict_sheets

## 4. Data format

In [174]:
# =============================================================================
# Section 4: Data Format
# =============================================================================

# =================== Account information =================== #

# Initialize a list of dataframes, since they have the same structure
list_df_payments = [df_AE,df_CL,df_VA]

# ======= Format Variables ======= #

# === Date Variables === #
cols_date = ['Cover period from', 'Cover period to','Claim Date','Claim Report', 'Date of payment','Payment request date']

# Iterate over the dataframes and columns: strip + convert to datetime (single pass)
for i in range(len(list_df_payments)):
    for col in cols_date:
        list_df_payments[i][col] = list_df_payments[i][col].astype(str).str.strip()
        list_df_payments[i][col] = pd.to_datetime(list_df_payments[i][col], dayfirst=True, errors='coerce')

# Validate that all date columns exist
for i, df in enumerate(list_df_payments):
    missing = [col for col in cols_date if col not in df.columns]
    if missing:
        print(f"❌ DataFrame {i} no tiene las columnas: {missing}")
    else:
        print(f"✅ DataFrame {i} tiene todas las columnas de fecha")

# === String Variables === #
cols_string = ['Claims Reference', 'Lob-Inward']
# Delete spaces; guard against str(NaN) becoming the literal "nan"
for i in range(len(list_df_payments)):
    for col in cols_string:
        list_df_payments[i][col] = list_df_payments[i][col].apply(
            lambda x: str(x).strip().replace(' ', '') if pd.notna(x) and str(x).strip().lower() != 'nan' else x
        )

# Clean 'Policy N°' to avoid mismatches in subsidiary filter
for i in range(len(list_df_payments)):
    list_df_payments[i]['Policy N°'] = list_df_payments[i]['Policy N°'].astype(str).str.strip()

# ======= Data Cleaning ======= #

cols_payment_base = ['Policy N°','Claims Reference','Claim Date','Claim Report','Lob-Inward','Currency','Payment request date','Date of payment','Amount USD (CORRECT)','Status']

# ========== FIX: Mapa canonico de normalizacion de Lob-Inward ==========
lob_normalize_map = {
    'cascoymaquinaria':   'CASCO Y MAQUINARIA',
    'cascoy maquinaria':  'CASCO Y MAQUINARIA',
    'casco y maquinaria': 'CASCO Y MAQUINARIA',   # variante que faltaba (mixed case)
    'casco':              'CASCO Y MAQUINARIA',
    'p&i':                'PANDI',
    'pandi':              'PANDI',
    'dw':                 'DEEP WATERS',
    'deep waters':        'DEEP WATERS',
    'deepwaters':         'DEEP WATERS',
    'cargo':              'CARGO',
    'plataformas':        'PLATAFORMAS',
    'floteles':           'FLOTELES',
}

for i in range(len(list_df_payments)):
    normalized = (
        list_df_payments[i]['Lob-Inward']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)   # collapse double spaces
        .map(lob_normalize_map)
    )
    # Only replace where a match was found; preserve original if not in map
    mask_mapped = normalized.notna()
    list_df_payments[i].loc[mask_mapped, 'Lob-Inward'] = normalized[mask_mapped]

# Audit: print unique LoB values post-normalization
print("\n=== Valores unicos de Lob-Inward post-normalizacion ===")
for name, df in [('df_AE', list_df_payments[0]), ('df_CL', list_df_payments[1]), ('df_VA', list_df_payments[2])]:
    print(f"\n{name}:")
    print(df['Lob-Inward'].value_counts(dropna=False).to_string())

# === Create KEY LOB: Claims Reference + Lob-Inward ===
for i in range(len(list_df_payments)):
    list_df_payments[i]['KEY LOB'] = (
        list_df_payments[i]['Claims Reference'].astype(str)
        + "-"
        + list_df_payments[i]['Lob-Inward'].astype(str)
    )

# Update dataframe references
df_AE = list_df_payments[0]
df_CL = list_df_payments[1]
df_VA = list_df_payments[2]

# Verify KEY LOB was created successfully
print("\n✓ Verificando que KEY LOB existe en todos los dataframes:")
for name, df in [('df_AE', df_AE), ('df_CL', df_CL), ('df_VA', df_VA)]:
    if 'KEY LOB' in df.columns:
        print(f"  ✓ {name} contiene KEY LOB")
    else:
        print(f"  ✗ {name} NO contiene KEY LOB")

# Validate: detect "nan" contamination in KEY LOB
print("\n=== Validacion: KEY LOBs con nan (contaminacion por NaN) ===")
for name, df in [('df_AE', df_AE), ('df_CL', df_CL), ('df_VA', df_VA)]:
    bad = df[df['KEY LOB'].str.contains('nan', case=False, na=False)]
    if len(bad) > 0:
        print(f"  ⚠️ {name}: {len(bad)} registros con nan en KEY LOB — revisar Claims Reference vacio")
        print(bad[['Claims Reference', 'Lob-Inward', 'KEY LOB', 'Amount USD (CORRECT)']].head())
    else:
        print(f"  ✅ {name}: sin contaminacion de nan")

# === Filtering by date ===
cols_payment = cols_payment_base + ['KEY LOB']
df_AE_filter = df_AE[(df_AE['Payment request date'] >= date_start_suma) | pd.isna(df_AE['Payment request date'])][cols_payment]
df_CL_filter = df_CL[(df_CL['Payment request date'] >= date_start_suma) | pd.isna(df_CL['Payment request date'])][cols_payment]
df_VA_filter = df_VA[(df_VA['Payment request date'] >= date_start_suma) | pd.isna(df_VA['Payment request date'])][cols_payment]

# NaT audit report
print("\n=== NaT en Payment request date (pasan el filtro por isna) ===")
for name, df in [('AE', df_AE), ('CL', df_CL), ('VA', df_VA)]:
    nat_count = df['Payment request date'].isna().sum()
    print(f"  {name}: {nat_count} NaT de {len(df)} registros ({nat_count/max(len(df),1):.1%})")

# === Filter by Lob-Inward Marine (only canonical values) ===
list_lob_marine = ['CASCO Y MAQUINARIA', 'CARGO', 'PANDI', 'DEEP WATERS', 'PLATAFORMAS', 'FLOTELES']

list_lob_nomarine = []
for df in [df_AE_filter, df_CL_filter, df_VA_filter]:
    valores_fuera = df[~df['Lob-Inward'].isin(list_lob_marine)]['Lob-Inward'].unique()
    list_lob_nomarine.extend(valores_fuera)
list_lob_nomarine = set(list_lob_nomarine)

print(f'\nLoB consideradas para Marine: {list_lob_marine}')
print(f'LoB NO consideradas para Marine: {list_lob_nomarine}')

# Apply marine filter
df_AE_filter = df_AE_filter[df_AE_filter['Lob-Inward'].isin(list_lob_marine)]
df_CL_filter = df_CL_filter[df_CL_filter['Lob-Inward'].isin(list_lob_marine)]
df_VA_filter = df_VA_filter[df_VA_filter['Lob-Inward'].isin(list_lob_marine)]

# === Filter by claims declared in BDX (Codigo 1 / df_update_db) ===
# Solo polizas vigentes (declaradas en el BDX del mes) deben generar
# movimientos financieros en la base final. Pagos de contabilidad sobre
# claims que no estan en el BDX (polizas no vigentes / fuera de cobertura)
# se EXCLUYEN, pero se reportan antes de descartarlos para no perder
# trazabilidad del monto.

# Normalizar CLAIM NUMBER del BDX igual que 'Claims Reference' (sin espacios)
claims_bdx = set(
    df_update_db['CLAIM NUMBER'].astype(str).str.strip().str.replace(' ', '')
)

print(f'\nClaims declarados en BDX (Codigo 1): {len(claims_bdx)}')

list_claims_huerfanos = set()
for nombre, df in [('AE', df_AE_filter), ('CL', df_CL_filter), ('VA', df_VA_filter)]:
    huerfanos = df.loc[~df['Claims Reference'].isin(claims_bdx), 'Claims Reference'].unique()
    if len(huerfanos) > 0:
        list_claims_huerfanos.update(huerfanos)
        monto = df.loc[df['Claims Reference'].isin(huerfanos), 'Amount USD (CORRECT)'].sum()
        print(f'  Warning {nombre}: {len(huerfanos)} claim(s) sin poliza vigente en BDX (${monto:,.2f}): {sorted(huerfanos)}')

print(f'\nTotal claims excluidos por no tener poliza vigente en BDX: {len(list_claims_huerfanos)}')

mask_huerfanos_cl = ~df_CL_filter['Claims Reference'].isin(claims_bdx)
df_CL_filter.loc[mask_huerfanos_cl, ['Claims Reference','Policy N°','Amount USD (CORRECT)']].to_excel(
    f'{ruta_procesados}/DEBUG_huerfanos_CL_{AñoMes}.xlsx', index=False
)
print(df_CL_filter.loc[mask_huerfanos_cl, 'Amount USD (CORRECT)'].sum())

if list_claims_huerfanos:
    rows = []
    for nombre, df in [('AE', df_AE_filter), ('CL', df_CL_filter), ('VA', df_VA_filter)]:
        sub = df[df['Claims Reference'].isin(list_claims_huerfanos)].copy()
        if len(sub) > 0:
            sub['ORIGEN'] = nombre
            rows.append(sub)
    df_huerfanos_export = pd.concat(rows, ignore_index=True)
    df_huerfanos_export.to_excel(f'{ruta_procesados}/CLAIMS_SIN_POLIZA_VIGENTE_{AñoMes}.xlsx', index=False)

# Aplicar el filtro: solo claims con poliza vigente (declarada en BDX)
df_AE_filter = df_AE_filter[df_AE_filter['Claims Reference'].isin(claims_bdx)]
df_CL_filter = df_CL_filter[df_CL_filter['Claims Reference'].isin(claims_bdx)]
df_VA_filter = df_VA_filter[df_VA_filter['Claims Reference'].isin(claims_bdx)]

# === Filter to exclude subsidiary company policies ===
list_subsidiary = ['25300 30028443', '25300 30031974']
subsidiary_clean = [s.strip() for s in list_subsidiary]

df_AE_filter = df_AE_filter[~df_AE_filter['Policy N°'].str.strip().isin(subsidiary_clean)]
df_CL_filter = df_CL_filter[~df_CL_filter['Policy N°'].str.strip().isin(subsidiary_clean)]
df_VA_filter = df_VA_filter[~df_VA_filter['Policy N°'].str.strip().isin(subsidiary_clean)]


# === Group by KEY LOB ===
df_AE_final = df_AE_filter.groupby(by=['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_AE_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative EXPENSES PAID {AñoMes}'}, inplace=True)

df_CL_final = df_CL_filter.groupby(by=['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_CL_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative CLAIMS PAID {AñoMes}'}, inplace=True)

df_VA_final = df_VA_filter.groupby(by=['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_VA_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative VALUATION EXPENSES {AñoMes}'}, inplace=True)


# === Final merge via concat + groupby ===
df_all = pd.concat([df_AE_final, df_CL_final, df_VA_final], ignore_index=True)
df_conta_final = (
    df_all
    .groupby('KEY LOB', as_index=False)
    .agg({
        f'Cumulative EXPENSES PAID {AñoMes}':      'sum',
        f'Cumulative CLAIMS PAID {AñoMes}':        'sum',
        f'Cumulative VALUATION EXPENSES {AñoMes}': 'sum',
        'Policy N°': 'first'
    })
)

# === Reconstruct LoB-Inward on df_conta_final ===
def consolidar_columna(df, base):
    """
    Consolida las variantes de una columna que quedan tras un merge
    (base_old/base_new o base_x/base_y) en una sola columna `base`,
    priorizando la version mas reciente (new/x) y usando la anterior
    (old/y) para rellenar los nulos. Tolera re-ejecuciones fuera de orden:
    si `base` ya existe sin sufijos, no hace nada.
    """
    for suf_new, suf_old in [('_new', '_old'), ('_x', '_y')]:
        col_new, col_old = f'{base}{suf_new}', f'{base}{suf_old}'
        if col_new in df.columns and col_old in df.columns:
            df[base] = df[col_new].fillna(df[col_old])
            df.drop(columns=[col_new, col_old], inplace=True)
        elif col_new in df.columns:
            df.rename(columns={col_new: base}, inplace=True)
        elif col_old in df.columns:
            df.rename(columns={col_old: base}, inplace=True)
    return df

df_conta_final['CLAIM NUMBER'] = df_conta_final['KEY LOB'].str.split('-', n=1).str[0]

df_update_db.columns = df_update_db.columns.str.strip()

for col_base in ['CLAIM NUMBER', 'LoB-Inward']:
    df_update_db = consolidar_columna(df_update_db, col_base)

df_lob_update = (
    df_update_db
    .drop_duplicates(subset='CLAIM NUMBER')
    .set_index('CLAIM NUMBER')['LoB-Inward']
)

df_conta_final['LoB-Inward'] = df_conta_final['CLAIM NUMBER'].map(df_lob_update)

df_lob_all = (
    df_all
    .drop_duplicates(subset='KEY LOB')
    .assign(LoB_Inward=lambda x: x['KEY LOB'].str.split('-', n=1).str[1])
    .set_index('KEY LOB')['LoB_Inward']
)

mask = df_conta_final['LoB-Inward'].isna()
df_conta_final.loc[mask, 'LoB-Inward'] = df_conta_final.loc[mask, 'KEY LOB'].map(df_lob_all)

df_conta_final['KEY LOB'] = (
    df_conta_final['CLAIM NUMBER'].astype(str)
    + "-"
    + df_conta_final['LoB-Inward'].astype(str)
)

# === Final format ===
for col in [f'Cumulative EXPENSES PAID {AñoMes}', f'Cumulative CLAIMS PAID {AñoMes}', f'Cumulative VALUATION EXPENSES {AñoMes}']:
    df_conta_final[col] = df_conta_final[col].fillna(0)

df_conta_final['FLAG CONTA'] = 1

# Print Validation
print(f"\nTenemos un total de ${round(df_conta_final[f'Cumulative EXPENSES PAID {AñoMes}'].sum(), 2)} en Cumulative EXPENSES PAID en el periodo")
print(f"Tenemos un total de ${round(df_conta_final[f'Cumulative CLAIMS PAID {AñoMes}'].sum(), 2)} en Cumulative CLAIMS PAID en el periodo")
print(f"Tenemos un total de ${round(df_conta_final[f'Cumulative VALUATION EXPENSES {AñoMes}'].sum(), 2)} en Cumulative VALUATION EXPENSES en el periodo")

# ===== Handmade Adjustments ===== #
if AñoMes == 202503:
    replacements = {
        '323372640000025-CARGO': '323372640000025-CARGA POLIETILENO',
        '323372640000051-CARGO': '323372640000051-CARGA POLIETILENO'
    }
    df_conta_final['KEY LOB'] = df_conta_final['KEY LOB'].replace(replacements)

cols = ["CLAIM NUMBER", "LoB-Inward", "Policy N°"]
df_conta_final = df_conta_final.drop(columns=cols, errors='ignore')


# =============================================================================
# Reconciliacion Payments: dfcontafinal (post-filtros) vs archivo Payments crudo
# =============================================================================
# Por cada subsidiaria (AE/CL/VA), compara el monto que SI llega a la base
# contable final (df_AE_final/df_CL_final/df_VA_final, ya filtrado por fecha,
# LoB Marine, BDX vigente y polizas subsidiarias excluidas) contra el monto
# crudo del archivo Payments_{AñoMes}.xlsx tal como se recibio, sin ningun
# filtro. Alimenta la hoja Payments del reporte Validacion_OSLR (Seccion 10).
def _reconciliar_payments_subsidiaria(df_final, df_crudo, col_monto_final, nombre_subsidiaria):
    '''
    Compara, por NUMERO DE SINIESTRO (Claims Reference), el monto sumado de
    una subsidiaria que SI llega a la base contable final contra el monto
    crudo del archivo Payments original.

    Parametros
    ----------
    df_final : pandas.DataFrame
        df_AE_filter / df_CL_filter / df_VA_filter (transacciones ya
        filtradas por fecha, LoB Marine, BDX-vigente y subsidiaria excluida;
        SIN agrupar todavia por KEY LOB) con las columnas 'Claims Reference'
        y col_monto_final.
    df_crudo : pandas.DataFrame
        df_AE_marine / df_CL_marine / df_VA_marine (Payments_{AñoMes}.xlsx
        restringido a fecha del periodo (>= date_start_suma, o NaT) + LoB
        Marine; BDX-vigente y poliza-subsidiaria-excluida siguen sin
        aplicarse a proposito) con columnas 'Claims Reference',
        'Amount USD (CORRECT)'.
    col_monto_final : str
        Nombre de la columna de monto en df_final (ej. 'Amount USD
        (CORRECT)' si df_final es df_*_filter, no la ya renombrada).
    nombre_subsidiaria : str
        'AE', 'CL' o 'VA'. Solo se usa para mensajes de error.

    Retorna
    -------
    pandas.DataFrame
        Columnas: Claims Reference, dfcontafinal, PAYMENTS AñoMes. Union
        (outer) de siniestros de ambas fuentes, ordenado por Claims
        Reference, montos faltantes rellenados con 0.

    Excepciones
    -----------
    KeyError
        Si a df_final le falta 'Claims Reference' o col_monto_final, o si a
        df_crudo le falta 'Claims Reference' o 'Amount USD (CORRECT)'.
    '''
    if 'Claims Reference' not in df_final.columns or col_monto_final not in df_final.columns:
        raise KeyError(
            f"[{nombre_subsidiaria}] df_final no tiene las columnas esperadas "
            f"('Claims Reference', '{col_monto_final}')."
        )
    if 'Claims Reference' not in df_crudo.columns or 'Amount USD (CORRECT)' not in df_crudo.columns:
        raise KeyError(
            f"[{nombre_subsidiaria}] df_crudo no tiene las columnas esperadas "
            "('Claims Reference', 'Amount USD (CORRECT)')."
        )

    final_siniestro = (
        df_final.groupby('Claims Reference', as_index=False)[col_monto_final]
        .sum()
        .rename(columns={col_monto_final: 'dfcontafinal'})
    )
    crudo_siniestro = (
        df_crudo.groupby('Claims Reference', as_index=False)['Amount USD (CORRECT)']
        .sum()
        .rename(columns={'Amount USD (CORRECT)': 'PAYMENTS AñoMes'})
    )

    reconciliacion = final_siniestro.merge(crudo_siniestro, on='Claims Reference', how='outer')
    reconciliacion[['dfcontafinal', 'PAYMENTS AñoMes']] = (
        reconciliacion[['dfcontafinal', 'PAYMENTS AñoMes']].fillna(0)
    )
    return reconciliacion.sort_values('Claims Reference').reset_index(drop=True)


cols_reconciliacion_vacio = ['Policy N°', 'dfcontafinal', 'PAYMENTS AñoMes']

# 'dfcontafinal' (df_AE_final/CL/VA) ya paso por TODOS los filtros de fecha y
# LoB de la Seccion 4 (fecha >= date_start_suma o NaT, LoB Marine), ademas de
# BDX-vigente y poliza-subsidiaria-excluida. El lado crudo debe pasar por el
# MISMO filtro de FECHA y LoB Marine antes de comparar -si no, cualquier claim
# de un periodo anterior (fecha < date_start_suma) o de otra LoB (ej. Autos,
# Property) se ve como "diferencia" en la hoja Payments sin serlo realmente,
# solo por no aplicar al mes de este AñoMes.
# NOTA (a proposito): BDX-vigente y poliza-subsidiaria-excluida siguen SIN
# aplicarse aqui, para que la hoja Payments SI detecte esos dos casos -un
# claim con fecha y LoB correctos para el periodo, pero excluido de
# dfcontafinal por no estar en el BDX vigente o por ser poliza subsidiaria,
# debe seguir viendose como diferencia a revisar-.
mask_fecha_AE = (df_AE['Payment request date'] >= date_start_suma) | pd.isna(df_AE['Payment request date'])
mask_fecha_CL = (df_CL['Payment request date'] >= date_start_suma) | pd.isna(df_CL['Payment request date'])
mask_fecha_VA = (df_VA['Payment request date'] >= date_start_suma) | pd.isna(df_VA['Payment request date'])

df_AE_marine = df_AE[mask_fecha_AE & df_AE['Lob-Inward'].isin(list_lob_marine)]
df_CL_marine = df_CL[mask_fecha_CL & df_CL['Lob-Inward'].isin(list_lob_marine)]
df_VA_marine = df_VA[mask_fecha_VA & df_VA['Lob-Inward'].isin(list_lob_marine)]
print(
    f'Payments crudo filtrado a fecha del periodo + LoB Marine: AE {len(df_AE_marine)}/{len(df_AE)} filas, '
    f'CL {len(df_CL_marine)}/{len(df_CL)} filas, VA {len(df_VA_marine)}/{len(df_VA)} filas.'
)

try:
    df_reconciliacion_AE = _reconciliar_payments_subsidiaria(
        df_AE_filter, df_AE_marine, 'Amount USD (CORRECT)', 'AE'
    )
    df_reconciliacion_CL = _reconciliar_payments_subsidiaria(
        df_CL_filter, df_CL_marine, 'Amount USD (CORRECT)', 'CL'
    )
    df_reconciliacion_VA = _reconciliar_payments_subsidiaria(
        df_VA_filter, df_VA_marine, 'Amount USD (CORRECT)', 'VA'
    )
    n_dif_AE = (df_reconciliacion_AE['dfcontafinal'].round(2) != df_reconciliacion_AE['PAYMENTS AñoMes'].round(2)).sum()
    n_dif_CL = (df_reconciliacion_CL['dfcontafinal'].round(2) != df_reconciliacion_CL['PAYMENTS AñoMes'].round(2)).sum()
    n_dif_VA = (df_reconciliacion_VA['dfcontafinal'].round(2) != df_reconciliacion_VA['PAYMENTS AñoMes'].round(2)).sum()
    print(
        f'Reconciliación Payments: AE {len(df_reconciliacion_AE)} siniestros ({n_dif_AE} con diferencia), '
        f'CL {len(df_reconciliacion_CL)} siniestros ({n_dif_CL} con diferencia), '
        f'VA {len(df_reconciliacion_VA)} siniestros ({n_dif_VA} con diferencia).'
    )

    # =========================================================================
    # GUARDIA PAYMENTS (nb2): el groupby por Policy N° (hoja Payments) reduce
    # el NUMERO DE FILAS frente a df_conta_final (agrupado por KEY LOB) cuando
    # varias claims comparten poliza -eso es normal y esperado-, pero el MONTO
    # TOTAL no se debe mover ni un centavo al reagrupar. Se compara la suma de
    # 'dfcontafinal' (por poliza) contra la suma de la columna equivalente en
    # df_conta_final (por KEY LOB); si no cuadran exacto, algo se perdio o se
    # duplico al reagrupar y se detiene antes de exportar la hoja Payments.
    # =========================================================================
    TOLERANCIA_RECONCILIACION = 0.01
    _totales_check = [
        ('AE', df_reconciliacion_AE['dfcontafinal'].sum(), df_conta_final[f'Cumulative EXPENSES PAID {AñoMes}'].sum()),
        ('CL', df_reconciliacion_CL['dfcontafinal'].sum(), df_conta_final[f'Cumulative CLAIMS PAID {AñoMes}'].sum()),
        ('VA', df_reconciliacion_VA['dfcontafinal'].sum(), df_conta_final[f'Cumulative VALUATION EXPENSES {AñoMes}'].sum()),
    ]
    for _sub, _total_payments, _total_conta in _totales_check:
        _dif = abs(_total_payments - _total_conta)
        if _dif > TOLERANCIA_RECONCILIACION:
            raise ValueError(
                f"GUARDIA PAYMENTS (nb2): [{_sub}] la suma de 'dfcontafinal' agrupada por Policy N° "
                f"(${_total_payments:,.2f}) no cuadra con la suma en df_conta_final agrupada por KEY LOB "
                f"(${_total_conta:,.2f}). Diferencia: ${_dif:,.2f}. Que baje el NUMERO de filas al "
                "agrupar por poliza es normal (varias claims comparten poliza), pero el MONTO total "
                "no deberia cambiar - revisar antes de exportar la hoja Payments."
            )
    print(
        f'✅ Guardia Payments: AE/CL/VA — suma por póliza cuadra exacto contra df_conta_final '
        f'(por KEY LOB), tolerancia ${TOLERANCIA_RECONCILIACION}.'
    )
except KeyError as e:
    print(f'Reconciliación Payments omitida (hoja Payments se exportará vacía): {e}')
    df_reconciliacion_AE = pd.DataFrame(columns=cols_reconciliacion_vacio)
    df_reconciliacion_CL = pd.DataFrame(columns=cols_reconciliacion_vacio)
    df_reconciliacion_VA = pd.DataFrame(columns=cols_reconciliacion_vacio)


✅ DataFrame 0 tiene todas las columnas de fecha
✅ DataFrame 1 tiene todas las columnas de fecha
✅ DataFrame 2 tiene todas las columnas de fecha

=== Valores unicos de Lob-Inward post-normalizacion ===

df_AE:
Lob-Inward
ONL                   20
ONPD                  18
NaN                    7
OFFPD                  4
PANDI                  2
PMI                    2
CASCO Y MAQUINARIA     1
OFFL                   1

df_CL:
Lob-Inward
CARGO    24
ONL      22
EE-T     18
NaN      11
ONPD      7
OFFL      5

df_VA:
Lob-Inward
NaN     8
ONL     3
ONPD    2

✓ Verificando que KEY LOB existe en todos los dataframes:
  ✓ df_AE contiene KEY LOB
  ✓ df_CL contiene KEY LOB
  ✓ df_VA contiene KEY LOB

=== Validacion: KEY LOBs con nan (contaminacion por NaN) ===
  ✅ df_AE: sin contaminacion de nan
  ✅ df_CL: sin contaminacion de nan
  ✅ df_VA: sin contaminacion de nan

=== NaT en Payment request date (pasan el filtro por isna) ===
  AE: 7 NaT de 55 registros (12.7%)
  CL: 11 NaT de 87 registros (

In [175]:
# --- Consolidación de FLAG CONTA tras posibles re-ejecuciones fuera de orden ---
df_update_db = consolidar_columna(df_update_db, 'FLAG CONTA')

print('Consolidación de FLAG CONTA completada. Columnas actuales:')
print([col for col in df_update_db.columns if 'FLAG CONTA' in col])

Consolidación de FLAG CONTA completada. Columnas actuales:
[]


## 5. Data Cross

In [176]:
# =============================================================================
# Section 5: Data Cross
# =============================================================================
print('=' * 95)
print('Comenzamos con el cruce entre la base actualizable del mes y la información contable del mes')
print(f'Tenemos {len(df_update_db)} registros en la base actualizable del mes')
print(f'Tenemos {len(df_conta_final)} registros en la base contable del mes')

# =============================================================================
# PASO 1: Consolidar columnas _x/_y remanentes de re-ejecuciones previas
# =============================================================================
for col_base in ['CLAIM NUMBER', 'LoB-Inward', 'FLAG CONTA']:
    df_update_db = consolidar_columna(df_update_db, col_base)

print(f"\n¿FLAG CONTA existe en df_update_db? {'FLAG CONTA' in df_update_db.columns}")

# =============================================================================
# PASO 2: Preparar y realizar el merge con la información contable
# =============================================================================
cols_extra = [
    'KEY LOB',
    'FLAG CONTA',
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative CLAIMS PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}'
]
cols_extra_ok = [c for c in cols_extra if c in df_conta_final.columns]
cols_missing = set(cols_extra) - set(cols_extra_ok)
if cols_missing:
    print(f"❌ Columnas faltantes en df_conta_final: {cols_missing}")

registros_antes = len(df_update_db)

df_update_db = df_update_db.merge(
    df_conta_final[cols_extra_ok],
    on='KEY LOB',
    how='outer',  # incluye registros nuevos que solo vienen de contabilidad
    suffixes=('_old', '_new')
)

print(f"\nRegistros antes del merge: {registros_antes}")
print(f"Registros después del merge: {len(df_update_db)}")
print(f"Incremento de registros: {len(df_update_db) - registros_antes}")

# =============================================================================
# PASO 3: Consolidar columnas duplicadas por el merge (_old/_new) y rellenar nulos
# =============================================================================
cols_to_consolidate = [
    'FLAG CONTA',
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative CLAIMS PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}'
]
for col_base in cols_to_consolidate:
    df_update_db = consolidar_columna(df_update_db, col_base)
    if col_base in df_update_db.columns:
        df_update_db[col_base] = df_update_db[col_base].fillna(0)

if not df_update_db.columns.is_unique:
    print("⚠️ Columnas duplicadas detectadas tras el merge. Corrigiendo...")
    df_update_db = df_update_db.loc[:, ~df_update_db.columns.duplicated()]

# Diagnóstico: mostrar registros con movimientos financieros del mes
nuevos_en_merge = df_update_db[df_update_db['FLAG CONTA'] == 1]
print(f"\nRegistros con movimientos financieros (FLAG CONTA=1): {len(nuevos_en_merge)}")


# =============================================================================
# PASO 4: Evitar doble conteo en KEY LOB con múltiples filas actuariales
#
# Entrada: df_update_db con 'KEY LOB', GROSS RESERVE/DEDUCTIBLE del mes, y las
#          columnas de cols_zerear (pagos acumulados del mes, provenientes del
#          merge outer con contabilidad en PASO 2).
# Salida: cols_zerear actualizadas in-place; no retorna valor.
# Excepciones: ninguna explicita — columnas ausentes se omiten (cols_extra_ok
#              ya filtro antes las que si llegaron del merge).
# =============================================================================
cols_zerear = [
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}',
    f'Cumulative CLAIMS PAID {AñoMes}',
]

# Criterio de calidad para desempatar entre filas duplicadas del mismo KEY LOB:
# 1) se prefiere la fila con GROSS RESERVE y DEDUCTIBLE no nulos (mas completa)
# 2) entre las completas, se prefiere la de mayor DEDUCTIBLE
def _mejores_filas_key_lob(df, anio_mes):
    completa = (
        df[f'GROSS RESERVE {anio_mes}'].notna() & df[f'DEDUCTIBLE {anio_mes}'].notna()
    ).astype(int)
    ranking = pd.DataFrame({
        'KEY LOB': df['KEY LOB'],
        '_completa': completa,
        '_deducible': df[f'DEDUCTIBLE {anio_mes}'].fillna(-1),
    }, index=df.index).sort_values(
        by=['KEY LOB', '_completa', '_deducible'], ascending=[True, False, False]
    )
    return ranking.groupby('KEY LOB').head(1).index

mejor_idx_keylob = _mejores_filas_key_lob(df_update_db, AñoMes)
dup_keylob = df_update_db.duplicated(subset=['KEY LOB'], keep=False) & ~df_update_db.index.isin(mejor_idx_keylob)

if dup_keylob.any():
    print(f'\n⚠️  KEY LOBs con múltiples filas actuariales: {dup_keylob.sum()}')
    grp_pagos = df_update_db.groupby('KEY LOB')
    # Por cada columna de pago: si el monto es igual en TODAS las filas del
    # KEY LOB, es el mismo dato contable duplicado por el merge (outer join
    # de PASO 2) -> se conserva una sola vez (NO se suma, sumar lo
    # multiplicaria por el numero de capas). Si difiere entre filas, son
    # montos distintos por capa/deducible -> SI se suman, para no perder
    # pago real de ninguna capa.
    for col in cols_zerear:
        if col not in df_update_db.columns:
            continue
        # nunique()+map() en vez de .transform(lambda...): esa forma puede
        # degradar a dtype float64 con NaN en ciertas versiones de pandas,
        # rompiendo el `~` booleano usado mas abajo.
        nunique_col = grp_pagos[col].nunique(dropna=False)
        mismo_valor = df_update_db['KEY LOB'].map(nunique_col) <= 1
        valor_unico = grp_pagos[col].transform('max')
        valor_sumado = grp_pagos[col].transform('sum')
        valor_final = valor_unico.where(mismo_valor, valor_sumado)

        n_sumados = int((~mismo_valor & dup_keylob).sum())
        if n_sumados:
            print(f"    · {col}: {n_sumados} fila(s) con monto distinto por capa -> sumado en la fila ganadora")

        df_update_db.loc[mejor_idx_keylob, col] = valor_final.loc[mejor_idx_keylob]
        df_update_db.loc[dup_keylob, col] = 0


# =============================================================================
# PASO 5: Rellenar registros nuevos que solo vienen de contabilidad
# =============================================================================
new_claims_mask = (
    (df_update_db['FLAG CONTA'] == 1) &
    (df_update_db[[f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}']].isna().any(axis=1))
)
new_claims_count = new_claims_mask.sum()
print(f"\nRegistros nuevos identificados (solo de contabilidad): {new_claims_count}")

if new_claims_count > 0:
    cols_fill_zero = [
        f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}',
        f'GROSS RESERVE {AñoMes_ant}', f'DEDUCTIBLE {AñoMes_ant}',
        f'Cumulative EXPENSES PAID {AñoMes_ant}', f'Cumulative CLAIMS PAID {AñoMes_ant}',
        f'Cumulative VALUATION EXPENSES {AñoMes_ant}', 'Cumulative SALVAGE',
        f'OSLR Inward {AñoMes_ant}'
    ]
    for col in cols_fill_zero:
        if col in df_update_db.columns:
            df_update_db.loc[new_claims_mask, col] = df_update_db.loc[new_claims_mask, col].fillna(0)

    text_defaults = {
        'STATUS': 'P',
        'PAYMENT STATUS': 'OPEN',
        'Monthly': 0,
        'NOTAS': 'Siniestro agregado desde contabilidad'
    }
    for col, default_val in text_defaults.items():
        if col in df_update_db.columns:
            df_update_db.loc[new_claims_mask, col] = df_update_db.loc[new_claims_mask, col].fillna(default_val)

    print("✓ Registros nuevos rellenados correctamente")

# =============================================================================
# PASO 6: Consolidar CLAIM NUMBER y LoB-Inward remanentes del merge
# =============================================================================
for col_base in ['CLAIM NUMBER', 'LoB-Inward']:
    df_update_db = consolidar_columna(df_update_db, col_base)

# =============================================================================
# PASO 7: Validación final
# =============================================================================
print('\n' + '=' * 95)
print('RESULTADO FINAL DEL CRUCE:')
print(f'Registros en df_update_db: {len(df_update_db)}')
columnas_criticas = ['KEY LOB', 'FLAG CONTA', 'CLAIM NUMBER', 'LoB-Inward']
faltantes = [c for c in columnas_criticas if c not in df_update_db.columns]
if faltantes:
    print(f'❌ Columnas críticas faltantes: {faltantes}')
else:
    registros_cruzados = df_update_db['FLAG CONTA'].sum()
    registros_sin_match = (df_update_db['FLAG CONTA'] == 0).sum()
    tasa_match = (registros_cruzados / len(df_update_db)) * 100 if len(df_update_db) > 0 else 0
    print(f'Registros que cruzaron: {registros_cruzados:,.0f}')
    print(f'Registros sin match: {registros_sin_match:,.0f}')
    print(f'Tasa de match: {tasa_match:.2f}%')

Comenzamos con el cruce entre la base actualizable del mes y la información contable del mes
Tenemos 4337 registros en la base actualizable del mes
Tenemos 25 registros en la base contable del mes

¿FLAG CONTA existe en df_update_db? False

Registros antes del merge: 4337
Registros después del merge: 4337
Incremento de registros: 0

Registros con movimientos financieros (FLAG CONTA=1): 25

⚠️  KEY LOBs con múltiples filas actuariales: 21
    · Cumulative EXPENSES PAID 202604: 3 fila(s) con monto distinto por capa -> sumado en la fila ganadora


    · Cumulative VALUATION EXPENSES 202604: 3 fila(s) con monto distinto por capa -> sumado en la fila ganadora
    · Cumulative CLAIMS PAID 202604: 3 fila(s) con monto distinto por capa -> sumado en la fila ganadora

Registros nuevos identificados (solo de contabilidad): 0

RESULTADO FINAL DEL CRUCE:
Registros en df_update_db: 4337
Registros que cruzaron: 25
Registros sin match: 4,312
Tasa de match: 0.58%


## 6. Creation of formulated variables

### Lógica de OSLR (3 pasos)

**1. Cálculo base (sin cambios)**

Se calcula `Cumulative CLAIMS PAID` (gastos + valuación + siniestros pagados, mes
anterior + mes actual) y luego el OSLR inicial:

```
OSLR = max( max(GROSS RESERVE − DEDUCTIBLE, 0) − Cumulative CLAIMS PAID, 0 )
```

⚠️ Excepción: las pólizas en `LIST_LEGACY` se excluyen de este cálculo y quedan intactas.

**2. NUEVO — Validación contra `Net Reserve (USD)` del BDX**

Inmediatamente después del cálculo base y antes de las 6 reglas de zereo.
Aplica solo a los registros **no legacy**, con `STATUS == 'P'` (abiertos) y **solo cuando
`Net Reserve (USD) == 0`**:

- Si `Net Reserve (USD) == 0` y el OSLR calculado no coincide (> tolerancia) → se fuerza el OSLR a 0.
- Si `Net Reserve (USD) != 0` (incluye negativos, ej. líneas CARGO) → el OSLR calculado se conserva
  intacto, para evitar introducir reservas negativas o ajustes no verificados.

Cada ajuste queda auditado en `Ajustes_OSLR_vs_NetReserve.xlsx` (registros afectados y $ de diferencia).

**3. Auditoría — Las 6 reglas de zereo**

Una vez ajustado el OSLR con la validación del BDX, se ejecutan en orden las 6 reglas.
Cada una reporta en consola cuántas filas afecta y cuánto OSLR elimina, y además
cada siniestro tocado queda registrado en `Log_Zereo_OSLR.xlsx` (CLAIM NUMBER, KEY LOB,
STATUS, regla, detalle, OSLR antes/después) — así se puede buscar un siniestro específico
directamente en Excel, sin escribir código, y ver qué regla le movió el OSLR:

1. **Ruido / Redondeo** — el OSLR remanente es insignificante (< `UMBRAL_OSLR_MINIMO`). Se zerea.
2. **Ajuste manual** — el claim está en `OSLR_ZERO_MANUAL` y el mes actual ≤ `periodo_hasta`. Se zerea.
3. **OSLR previo y esperado en cero** — el OSLR del mes pasado fue 0 y el recálculo con variables del mes pasado también da 0. Se zerea (evita "resurrecciones" artificiales).
4. **Legacy nulo** — pólizas de `LIST_LEGACY` con `STATUS == "P"` cuyo OSLR base quedó nulo/vacío. Se fuerza a cero.
5. **Sin cambios en el mes** — el claim está en `list_exceptions_monthly`, `FLAG CONTA == 0`, no es `Monthly`, y ni `GROSS RESERVE` ni `DEDUCTIBLE` variaron vs. el mes pasado. Se zerea el actual y se preserva el OSLR del mes anterior.
6. **KEY LOB duplicado** — el mismo `KEY LOB` aparece en múltiples filas (ej. multi-deducible). Se compara `GROSS RESERVE`/`DEDUCTIBLE` entre las filas: si son iguales en todas, es un duplicado real del merge y se conserva el OSLR solo en la fila mas completa/mayor DEDUCTIBLE (se zerea en las demás); si difieren, son capas de deducible distintas y se suma el OSLR de todas las filas en la fila ganadora (misma funcion `_mejores_filas_key_lob` usada en PASO 4).

In [177]:
# =============================================================================
# Section 6: Creation of formulated variables
# =============================================================================

# ======= Numerical variables ======= #

# == Exceptions == #
# OSLR
# List of records for wich we use the Gross Reserve and Deductible from previous month
list_ded_reserve = df_update_db[
(df_update_db['NOTAS'] != '')| # All the records that have a commment at this point
(df_update_db['Monthly'] == 1) # All the records that are monthly 
]['CLAIM NUMBER'].unique()

# Initialize the list of claim numbers with incidences during the month
list_exceptions_monthly = df_update_db[
(df_update_db['NOTAS'] != '') & # The claim number that has a comment in 'NOTAS'
(~df_update_db['NOTAS'].isin(['LEGACY INFO', 'LEGACY POLICY'])) # but the comment is different from 'LEGACY INFO', 'LEGACY POLICY'
]['CLAIM NUMBER'].unique()  

# Fill the current values of gross reserve and deductible with the values from previous month 
for col in [f'GROSS RESERVE {AñoMes_ant}', f'DEDUCTIBLE {AñoMes_ant}']:
    target_col = col.replace(str(AñoMes_ant), str(AñoMes))  # Update target column name dynamically
    df_update_db.loc[
        df_update_db['CLAIM NUMBER'].isin(list_ded_reserve),  # If the record is in the list 'list_ded_reserve'
        target_col                                                  # Column to be updated
    ] = df_update_db[col]                                           # Value from the previous month

# Expresión regular para eliminar caracteres ilegales para Excel
def clean_excel_string(s):
    if isinstance(s, str):
        # Quita caracteres no imprimibles ni válidos en XML
        return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', s)
    return s

# Identificar columnas numéricas
numeric_cols = df_update_db.select_dtypes(include=['number']).columns.tolist()
string_cols = df_update_db.select_dtypes(include=['object']).columns.tolist()
print(f"Columnas numéricas: {len(numeric_cols)}")
print(f"Columnas de texto: {len(string_cols)}")

# Aplicar limpieza SOLO a columnas de texto
for col in string_cols:
    df_update_db[col] = df_update_db[col].apply(clean_excel_string)
print("✓ Limpieza completada solo en columnas de texto")

# Debugg
df_update_db.to_excel(f'{ruta_procesados}/pruebas/actuarial.xlsx', index=False)

Columnas numéricas: 17
Columnas de texto: 20
✓ Limpieza completada solo en columnas de texto


C:\Users\IKAL14\AppData\Local\Temp\ipykernel_49192\4254446606.py:38: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_update_db.select_dtypes(include=['object']).columns.tolist()


In [178]:
# ============== Payments columns ============== #
# == Creation of formulated columns == #
# Agregamos .fillna(0) para que si no hay histórico del mes anterior, se asuma 0 y no destruya la suma.
df_update_db['Cumulative EXPENSES PAID'] = df_update_db[f'Cumulative EXPENSES PAID {AñoMes_ant}'].fillna(0) + df_update_db[f'Cumulative EXPENSES PAID {AñoMes}'].fillna(0)
df_update_db['Cumulative VALUATION EXPENSES'] = df_update_db[f'Cumulative VALUATION EXPENSES {AñoMes_ant}'].fillna(0) + df_update_db[f'Cumulative VALUATION EXPENSES {AñoMes}'].fillna(0)
df_update_db['Cumulative CLAIMS PAID'] = df_update_db[f'Cumulative CLAIMS PAID {AñoMes_ant}'].fillna(0) + df_update_db[f'Cumulative CLAIMS PAID {AñoMes}'].fillna(0)
df_update_db['Total Claims Paid Inward'] = df_update_db['Cumulative EXPENSES PAID'] + df_update_db['Cumulative VALUATION EXPENSES'] + df_update_db['Cumulative CLAIMS PAID'] + df_update_db['Cumulative SALVAGE'].fillna(0)
df_update_db['Total Claims Paid no Alae'] = df_update_db['Total Claims Paid Inward'] - df_update_db['Cumulative EXPENSES PAID']

# ============== OSLR Calculation ============== #
# Paso 1: OSLR = max(max(GR - DED, 0) - Cumulative CLAIMS PAID, 0)
# Paso 2: Validación contra Net Reserve (USD) del BDX 
# Paso 3: 6 reglas de zereo

oslr_col = f'OSLR Inward {AñoMes}'

# 1. Asegurar que todo sea string y sin espacios
df_update_db['INWARD POLICY N°'] = df_update_db['INWARD POLICY N°'].astype(str).str.strip()
LEGACY = [str(p).strip() for p in LIST_LEGACY]

# 2. Hacemos el cálculo matemático primero
calculo_oslr = np.maximum(
    np.maximum(df_update_db[f'GROSS RESERVE {AñoMes}'].fillna(0) - df_update_db[f'DEDUCTIBLE {AñoMes}'].fillna(0), 0)
    - df_update_db['Cumulative CLAIMS PAID'],
    0
)

# 3. Asignamos el resultado SOLO a los que NO son Legacy usando la máscara (.loc)
mask_no_legacy = ~df_update_db['INWARD POLICY N°'].isin(LEGACY)
df_update_db.loc[mask_no_legacy, oslr_col] = calculo_oslr

# =============================================================================
# Paso 2: Validación contra Net Reserve (USD) del BDX original
# Se aplica exclusivamente a los registros NO legacy, con STATUS == 'P' (abiertos)
# y SOLO cuando Net Reserve (USD) == 0. Si Net Reserve != 0 (incluye negativos,
# el OSLR calculado se conserva sin tocar, para evitar introducir reservas negativas.
# Se ejecuta ANTES de las 6 reglas de zereo.
# =============================================================================
TOLERANCIA_NET_RESERVE = 0.10  # tolerancia por redondeo, en USD
net_reserve_col = 'Net Reserve (USD)'
if net_reserve_col not in df_update_db.columns:
    raise KeyError(
        f"No se encontró la columna '{net_reserve_col}' en df_update_db. "
        "Verifica que el BDX incluya esta columna antes de continuar."
    )
df_update_db[net_reserve_col] = df_update_db[net_reserve_col].fillna(0)

# Restringe la validación a NO legacy, STATUS abierto ('P'/'p'), y Net Reserve == 0
mask_validacion_net_reserve = (
    mask_no_legacy
    & df_update_db['STATUS'].isin(['P', 'p'])
    & (df_update_db[net_reserve_col].abs() <= TOLERANCIA_NET_RESERVE)
)

dif_net_reserve = (df_update_db[oslr_col] - df_update_db[net_reserve_col]).abs()
mask_dif_net_reserve = mask_validacion_net_reserve & (dif_net_reserve > TOLERANCIA_NET_RESERVE)

n_dif_net_reserve = int(mask_dif_net_reserve.sum())
monto_ajuste_net_reserve = float(
    (df_update_db.loc[mask_dif_net_reserve, net_reserve_col]
     - df_update_db.loc[mask_dif_net_reserve, oslr_col]).sum()
)

print('\n' + '=' * 95)
print('VALIDACIÓN CONTRA NET RESERVE (USD) DEL BDX (solo Net Reserve == 0)')
print('=' * 95)
print(f"  Registros no-legacy, STATUS='P' y Net Reserve==0 validados: {int(mask_validacion_net_reserve.sum())}")
print(f"  Registros donde OSLR calculado != 0 (se fuerzan a 0): {n_dif_net_reserve}")
print(f"  Ajuste neto aplicado: ${monto_ajuste_net_reserve:,.2f}")

# Log de auditoría: qué se sobreescribió y por cuánto
df_ajustes_net_reserve = df_update_db.loc[mask_dif_net_reserve, [
    'CLAIM NUMBER', 'KEY LOB', oslr_col, net_reserve_col
]].copy()
df_ajustes_net_reserve.rename(columns={
    oslr_col: 'OSLR CALCULADO',
    net_reserve_col: 'NET RESERVE (USD)'
}, inplace=True)
df_ajustes_net_reserve['DIFERENCIA'] = (
    df_ajustes_net_reserve['NET RESERVE (USD)'] - df_ajustes_net_reserve['OSLR CALCULADO']
)
df_ajustes_net_reserve.to_excel(f'{ruta_incidencias}/Ajustes_OSLR_vs_NetReserve.xlsx', index=False)
print(f"  Log exportado: Ajustes_OSLR_vs_NetReserve.xlsx")
print('=' * 95 + '\n')

# Sobreescribir OSLR con el valor exacto de Net Reserve (USD) donde difieren
df_update_db.loc[mask_dif_net_reserve, oslr_col] = df_update_db.loc[mask_dif_net_reserve, net_reserve_col]

# =============================================================================
# LOG DE IMPACTO POR REGLA DE ZEREO
# Cada regla que pone OSLR en 0 reporta cuántos registros y cuánto dinero
# elimina, ANTES de aplicar el zereo — para que el proceso sea auditable.
# =============================================================================
resumen_zereo_oslr = []
log_zereo_oslr_detalle = []  # Detalle por siniestro/regla, para Log_Zereo_OSLR.xlsx

def _zerear_con_log(mask, regla, detalle=''):
    """Zerea oslr_col donde mask es True y registra el impacto (n registros, $ eliminado).
    Ademas guarda una fila por cada siniestro afectado, para poder auditar en Excel
    (sin escribir codigo) que regla movio el OSLR de un siniestro especifico."""
    n = int(mask.sum())
    monto = float(df_update_db.loc[mask, oslr_col].fillna(0).sum())
    resumen_zereo_oslr.append({'REGLA': regla, 'DETALLE': detalle, 'N REGISTROS': n, 'OSLR ELIMINADO': round(monto, 2)})
    print(f"  [{regla}] {n} registro(s) — ${monto:,.2f} de OSLR eliminado. {detalle}")
    if n > 0:
        cols_detalle = [c for c in ['CLAIM NUMBER', 'KEY LOB', 'STATUS','INWARD POLICY N°'] if c in df_update_db.columns]
        detalle_filas = df_update_db.loc[mask, cols_detalle].copy()
        detalle_filas['REGLA'] = regla
        detalle_filas['DETALLE'] = detalle
        detalle_filas['OSLR ANTES'] = df_update_db.loc[mask, oslr_col].fillna(0).values
        detalle_filas['OSLR DESPUES'] = 0.0
        log_zereo_oslr_detalle.append(detalle_filas)
        df_update_db.loc[mask, oslr_col] = 0

print('\n' + '=' * 95)
print('REGLAS DE ZEREO DE OSLR (con impacto en $):')
print('=' * 95)

# Regla 1: OSLR por debajo del umbral mínimo (ruido/redondeo)
UMBRAL_OSLR_MINIMO = 100
OSLR_ZERO_MANUAL = {
    '4431/2023' : 202509,
    '1560/2024' : 202509,
    '1563/2024' : 202509,
    '4478/2024' : 202509,
    '4801/2024' : 202509,
    '4803/2024' : 202509,
    '5704/2024' : 202509,
    '6399/2024' : 202509,
    '6400/2024' : 202509,
    '161/2025'  : 202509,
    '594/2025'  : 202509,
}

mask_umbral = df_update_db[oslr_col] < UMBRAL_OSLR_MINIMO
_zerear_con_log(mask_umbral, 'Umbral mínimo', f'OSLR < {UMBRAL_OSLR_MINIMO}')

# ====================== Handmade Adjustments ====================== #
# Regla 2: Claims zereados manualmente, con vigencia (config_marine.OSLR_ZERO_MANUAL)
def claims_oslr_zero_vigentes(AñoMes):
    """Devuelve los claims cuyo zereo manual de OSLR sigue vigente para AñoMes."""
    return [claim for claim, periodo_hasta in OSLR_ZERO_MANUAL.items() if AñoMes <= periodo_hasta]

claims_zero_vigentes = claims_oslr_zero_vigentes(AñoMes)
mask_manual = df_update_db['CLAIM NUMBER'].isin(claims_zero_vigentes)
_zerear_con_log(mask_manual, 'Ajuste manual', f'{len(claims_zero_vigentes)} claim(s) vigente(s) en config_marine.OSLR_ZERO_MANUAL')

#  ================================================================== #
# Regla 3: OSLR nulo en registros legacy pendientes
mask_legacy_null = (
    df_update_db['INWARD POLICY N°'].isin(LEGACY) & df_update_db['STATUS'].isin(['P', 'p']) & df_update_db[oslr_col].isnull()
)
_zerear_con_log(mask_legacy_null, 'Legacy nulo', 'poliza legacy, STATUS=P, OSLR nulo')

#  ================================================================== #
# Regla 4: KEY LOB duplicado (multi-deducible en el BDX)
#
# Criterio: para cada KEY LOB con filas duplicadas se compara GROSS RESERVE y
# DEDUCTIBLE del mes entre esas filas.
#   - Si coinciden en TODAS las filas -> duplicado real (artefacto del outer
#     join): se conserva el OSLR solo en la fila mas completa/mayor deducible
#     (via _mejores_filas_key_lob) y se zerea en las demas. Sumar aqui
#     inflaria el OSLR con el mismo monto repetido.
#   - Si difieren -> capas de deducible legitimamente distintas del mismo
#     claim (multi-deducible real del BDX): se SUMA el OSLR de todas las
#     filas del KEY LOB y el total se asigna a la fila mas completa/mayor
#     deducible; las demas se zerean para no duplicar el monto en el total.
#
# Entrada: df_update_db con 'KEY LOB', oslr_col, GROSS RESERVE/DEDUCTIBLE del mes.
# Salida: oslr_col actualizado in-place; no retorna valor.
# Excepciones: KeyError si faltan las columnas de GROSS RESERVE/DEDUCTIBLE del
#              mes (se detiene el proceso, no se hace suposicion silenciosa).
col_reserva, col_deducible = f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}'
faltantes_r4 = [c for c in (col_reserva, col_deducible) if c not in df_update_db.columns]
if faltantes_r4:
    raise KeyError(f"Regla 4 (KEY LOB duplicado): faltan columnas requeridas: {faltantes_r4}")

grp_keylob = df_update_db.groupby('KEY LOB')
es_dup_keylob = df_update_db.duplicated(subset=['KEY LOB'], keep=False)

# True si GROSS RESERVE y DEDUCTIBLE son iguales (o todos nulos) en TODO el grupo.
# nunique()+map() en vez de .transform(lambda...): esa forma puede degradar a
# dtype float64 con NaN en ciertas versiones de pandas, rompiendo el `~`
# booleano usado mas abajo (mismo problema visto en PASO 4).
nunique_gr = grp_keylob[col_reserva].nunique(dropna=False)
nunique_ded = grp_keylob[col_deducible].nunique(dropna=False)
mismo_gr_ded = (
    (df_update_db['KEY LOB'].map(nunique_gr) <= 1)
    & (df_update_db['KEY LOB'].map(nunique_ded) <= 1)
)
mask_dup_multideducible = es_dup_keylob & ~mismo_gr_ded  # GR/DED distintos -> sumar

mejor_idx_keylob_oslr = _mejores_filas_key_lob(df_update_db, AñoMes)

# Multi-deducible real: sumar el OSLR de todas las capas y asignarlo a la fila ganadora
idx_multideducible_mejor = mejor_idx_keylob_oslr.intersection(
    df_update_db.index[mask_dup_multideducible]
)
if len(idx_multideducible_mejor) > 0:
    total_oslr_keylob = grp_keylob[oslr_col].transform('sum')
    df_update_db.loc[idx_multideducible_mejor, oslr_col] = total_oslr_keylob.loc[idx_multideducible_mejor]

# Filas perdedoras (duplicado real o multi-deducible): se zerean
mask_dup_keylob_oslr = es_dup_keylob & ~df_update_db.index.isin(mejor_idx_keylob_oslr)
_zerear_con_log(
    mask_dup_keylob_oslr,
    'KEY LOB duplicado',
    'si GR y DEDUCTIBLE coinciden se conserva solo la fila mas completa/mayor deducible; '
    'si difieren (multi-deducible) se suma el OSLR de todas las capas en esa fila y se zerea en las demas'
)

#Regla 5: OSLR 0 en registros vacíos restantes sin importar legacy o status
mask_nulos_restantes = df_update_db[oslr_col].isna()
df_update_db.loc[mask_nulos_restantes, oslr_col] = 0
_zerear_con_log(mask_nulos_restantes, 'Zerar OSLR en registros nulos', 'OSLR nulos cambiados a 0')

# =============================================================================
# Exportar log detallado de zereo de OSLR (por siniestro y regla)
# Este archivo permite buscar un CLAIM NUMBER especifico en Excel y ver que regla(s) le
# modificaron el OSLR.
# =============================================================================
if log_zereo_oslr_detalle:
    df_log_zereo_oslr = pd.concat(log_zereo_oslr_detalle, ignore_index=True)
else:
    df_log_zereo_oslr = pd.DataFrame(columns=['CLAIM NUMBER', 'KEY LOB', 'STATUS', 'REGLA', 'DETALLE', 'OSLR ANTES', 'OSLR DESPUES'])
df_log_zereo_oslr.to_excel(f'{ruta_incidencias}/Log_Zereo_OSLR.xlsx', index=False)
n_claims_unicos = df_log_zereo_oslr['CLAIM NUMBER'].nunique() if len(df_log_zereo_oslr) else 0
print(f"\nLog detallado exportado: Log_Zereo_OSLR.xlsx ({len(df_log_zereo_oslr)} fila(s), {n_claims_unicos} siniestro(s) unico(s))")




# =============================================================================
# GUARDIA DE PROTECCION LEGACY (paso 1/2): congelar Reserva/Deducible/OSLR
# =============================================================================
# Blindaje estructural: sin importar que haya pasado en el codigo de arriba
# (calculo de OSLR, reglas de zereo, etc.), las polizas legacy SIEMPRE
# terminan aqui con Reserva/Deducible congelados en el valor del mes anterior
# y OSLR en 0. Los pagos contables (Cumulative EXPENSES/CLAIMS/VALUATION PAID)
# NO se tocan en esta guardia - siguen fluyendo normalmente para legacy,
# porque son movimientos de dinero reales que se siguen registrando aunque
# la poliza ya no reciba BDX nuevo.
mask_legacy_guard = df_update_db['INWARD POLICY N°'].isin(LEGACY)

df_update_db.loc[mask_legacy_guard, f'GROSS RESERVE {AñoMes}'] = \
    df_update_db.loc[mask_legacy_guard, f'GROSS RESERVE {AñoMes_ant}']
df_update_db.loc[mask_legacy_guard, f'DEDUCTIBLE {AñoMes}'] = \
    df_update_db.loc[mask_legacy_guard, f'DEDUCTIBLE {AñoMes_ant}']
df_update_db.loc[mask_legacy_guard, oslr_col] = 0

# Snapshot para re-validar despues de la Seccion 8 (rename + fillna podrian
# tocarlos sin querer si alguien modifica ese codigo en el futuro).
# Se usa el indice propio del DataFrame (no KEY LOB) porque KEY LOB puede
# estar legitimamente duplicado (multi-deducible - ver Regla 4 de zereo) y un
# merge por KEY LOB inflaria el conteo de filas de forma espuria. El indice
# se mantiene estable entre la Seccion 6 y la Seccion 8: no hay ningun drop
# ni filtro de filas en el camino, solo de columnas.
df_legacy_guard_snapshot = df_update_db.loc[
    mask_legacy_guard, [f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}', oslr_col]
].copy()
df_legacy_guard_snapshot.columns = ['GROSS RESERVE', 'DEDUCTIBLE', 'OSLR Inward']

print(f"\u2705 Guardia legacy (paso 1/2): {int(mask_legacy_guard.sum())} registros con Reserva/Deducible/OSLR congelados.")

# ============== INCURRED ============== #
# Al haber limpiado los NaN de OSLR y de Pagos, estas dos columnas saldrán perfectas y 100% numéricas
df_update_db['Inward Incurred'] = df_update_db['Total Claims Paid Inward'] + df_update_db[oslr_col]
df_update_db['Inward Incurred no Alae'] = df_update_db['Inward Incurred'] - df_update_db['Cumulative EXPENSES PAID']




# ======= Other variables ======= #
df_update_db['YEAR OF THE RESERVE'] = date_start_AñoMes.year
df_update_db['ACCIDENT YEAR'] = df_update_db['DATE OF LOSS'].dt.year
df_update_db['CLAIMS-ID'] = df_update_db['CLAIM NUMBER ORIGINAL'].astype(str) + df_update_db['LoB-Inward BDX'].astype(str)
df_update_db['ATTRITIONAL/LARGE'] = 'ATTRITIONAL'
df_update_db['INWARD POLICY'] = '02-Marine'
df_update_db['StandRe LoB'] = '03-MAT'
df_update_db['StandRe detailed LoB'] = 'Combined marine'
df_update_db['UW Year'] = np.where(
    df_update_db['DATE OF LOSS'].dt.month < 7,
    df_update_db['DATE OF LOSS'].dt.year - 1,
    df_update_db['DATE OF LOSS'].dt.year
)

# Guardar cambios limpios
# df_update_db.to_excel(f'C:/Users/IKAL14/Desktop/CALCULOS.xlsx', index= False)


VALIDACIÓN CONTRA NET RESERVE (USD) DEL BDX (solo Net Reserve == 0)
  Registros no-legacy, STATUS='P' y Net Reserve==0 validados: 2097
  Registros donde OSLR calculado != 0 (se fuerzan a 0): 1141
  Ajuste neto aplicado: $-3,846,532.27
  Log exportado: Ajustes_OSLR_vs_NetReserve.xlsx


REGLAS DE ZEREO DE OSLR (con impacto en $):
  [Umbral mínimo] 3288 registro(s) — $0.02 de OSLR eliminado. OSLR < 100
  [Ajuste manual] 0 registro(s) — $0.00 de OSLR eliminado. 0 claim(s) vigente(s) en config_marine.OSLR_ZERO_MANUAL
  [Legacy nulo] 629 registro(s) — $0.00 de OSLR eliminado. poliza legacy, STATUS=P, OSLR nulo
  [KEY LOB duplicado] 21 registro(s) — $0.00 de OSLR eliminado. si GR y DEDUCTIBLE coinciden se conserva solo la fila mas completa/mayor deducible; si difieren (multi-deducible) se suma el OSLR de todas las capas en esa fila y se zerea en las demas
  [Zerar OSLR en registros nulos] 308 registro(s) — $0.00 de OSLR eliminado. OSLR nulos cambiados a 0

Log detallado exportado: Log_Zereo_

## 7. OSLR and Dates Validation

In [179]:
# =============================================================================
# Section 7: OSLR and Dates Validation
# =============================================================================
print('\n' + '='*95)
print('SECCIÓN 7: VALIDACIONES DE OSLR Y FECHAS')
print('='*95)

# Identificar registros nuevos (solo de contabilidad) ANTES de hacer validaciones
new_records_mask = (df_update_db['FLAG CONTA'] == 1) & (df_update_db['NOTAS'] == 'Siniestro agregado desde contabilidad')
new_records_count = new_records_mask.sum()

print(f"\n📌 Registros nuevos (excluidos de validaciones estrictas): {new_records_count}")
if new_records_count > 0:
    print(f"   Estos registros se mantienen sin validación de fechas/OSLR")
    new_records_list = df_update_db.loc[new_records_mask, 'KEY LOB'].unique()
    for claim_key in new_records_list:
        print(f"   - {claim_key}")

# ======= Validations ======= #

# == Validation of 'OSLR Inward == #
# DIF OSLR se mantiene como float; la comparación usa tolerancia explícita
# (dif.abs() > 1) en vez de truncar a int
df_update_db['DIF OSLR'] = (df_update_db[f'OSLR Inward {AñoMes}'] - df_update_db[f'OSLR Inward {AñoMes_ant}']).fillna(0)

# Filter the records that meet the following conditions
# EXCLUIR registros nuevos de esta validación
df_errors_oslr = df_update_db[
    (df_update_db['DIF OSLR'].abs() > 1) & # Exist a difference of OSLR within months (tolerancia explícita)
    (df_update_db['FLAG CONTA'] == 0) & # The records doesn't have accounting movements in the month of process
    (df_update_db['Monthly'] == 0) & # The record is not a new claim
    ~(((df_update_db[f'GROSS RESERVE {AñoMes}'] - df_update_db[f'GROSS RESERVE {AñoMes_ant}']) != 0) | #Excludes records that had
     ((df_update_db[f'DEDUCTIBLE {AñoMes}'] - df_update_db[f'DEDUCTIBLE {AñoMes_ant}']) != 0)) &  # a change in Gross Reserve or Deductible from the previous month.
    ~new_records_mask  # EXCLUIR registros nuevos
]

# === Exportar incidencias de OSLR (bug #1: nunca se exportaban) ===
cols_errors_oslr = [
    'KEY LOB',
    f'OSLR Inward {AñoMes_ant}', f'OSLR Inward {AñoMes}', 'DIF OSLR',
    f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}', 'FLAG CONTA'
]
cols_errors_oslr = [c for c in cols_errors_oslr if c in df_errors_oslr.columns]
df_errors_oslr[cols_errors_oslr].to_excel(f'{ruta_incidencias}/Incidencias_OSLR.xlsx', index=False)
print(f'Se exportó correctamente el archivo de incidencias de OSLR ({len(df_errors_oslr)} registro(s))')

# == Validation of Dates == #
# EXCLUIR registros nuevos (que no tienen fechas)
df_errors_dates = df_update_db[
    (df_update_db['DATE OF LOSS'] < df_update_db['POLICY PERIOD START DATE']) |
    (df_update_db['DATE OF LOSS'] > df_update_db['POLICY PERIOD END DATE']) |
    ((df_update_db['DATE OF LOSS'] > df_update_db['DATE OF REPORT']) &
    (df_update_db['STATUS'] != 'C')) &
    ~new_records_mask  # EXCLUIR registros nuevos
].copy()

# Sanitize status
df_errors_dates.loc[:, 'STATUS'] = df_errors_dates['STATUS'].astype(str).str.strip().str.capitalize()

# Drop canceled and NaN records from this filtered DataFrame
df_errors_dates = df_errors_dates[~df_errors_dates['STATUS'].isin(['C','Nan',''])]

# Report
print('==============================================================================')
if not df_errors_dates.empty:
    print('Los registros con errores en fechas son los siguientes:')
    for index, row in df_errors_dates.iterrows():
        print(
            f"\nCLAIM NUMBER: {row['CLAIM NUMBER']}\n"
            f"  DATE OF LOSS          : {row['DATE OF LOSS']}\n"
            f"  DATE OF REPORT        : {row['DATE OF REPORT']}\n"
            f"  POLICY PERIOD START   : {row['POLICY PERIOD START DATE']}\n"
            f"  POLICY PERIOD END     : {row['POLICY PERIOD END DATE']}\n"
            f"  STATUS                : {row['STATUS']}\n"
        )
else:
    print("No se encontraron errores en fechas.")
print('==============================================================================')

# Final selection of columns
df_errors_dates = df_errors_dates[['CLAIM NUMBER ORIGINAL','DATE OF LOSS','DATE OF REPORT',
                                   'INWARD POLICY N°','LoB-Inward','POLICY PERIOD START DATE',
                                   'POLICY PERIOD END DATE','STATUS', 'NOTAS']]

# Create the error column according to the type of error
# Define the conditions and its value
conditions = [
    df_errors_dates['DATE OF LOSS'] < df_errors_dates['POLICY PERIOD START DATE'],
    df_errors_dates['DATE OF LOSS'] > df_errors_dates['POLICY PERIOD END DATE'],
    df_errors_dates['DATE OF LOSS'] > df_errors_dates['DATE OF REPORT']
]

# Define the choices
choices = ['DOL antes vigencia', 'DOL después vigencia', 'DOL después reporte']

# Create the column 'ERROR' based on the conditions
df_errors_dates['ERROR'] = np.select(conditions, choices, default='Sin error')
# Exportof the file
df_errors_dates.to_excel(f'{ruta_incidencias}/Incidencias_fechas.xlsx' ,index= False)
print('Se exportó correctamente el archivo de incidencias de fechas')
print('==============================================================================')

# LoB normalization
# Convert the relevant columns to string type
df_update_db[['LoB-Inward BDX', 'LoB-Inward','CLAIM NUMBER ORIGINAL']] = df_update_db[['LoB-Inward BDX', 'LoB-Inward','CLAIM NUMBER ORIGINAL']].astype(str)

# Replace 'nan' strings with empty strings in 'LoB-Inward BDX'
df_update_db['LoB-Inward BDX'] = df_update_db['LoB-Inward BDX'].replace('nan', '')

# Fill empty values in 'LoB-Inward BDX' with the values from 'LoB-Inward'
df_update_db.loc[df_update_db['LoB-Inward BDX'] == '', 'LoB-Inward BDX'] = df_update_db['LoB-Inward']
df_update_db['CLAIMS-ID'] = df_update_db['CLAIM NUMBER ORIGINAL'] + "-" + df_update_db['LoB-Inward BDX']

# Debugg
#df_update_db.to_excel(f'{ruta_procesados}/pruebas/actuarial.xlsx', index= False )


SECCIÓN 7: VALIDACIONES DE OSLR Y FECHAS

📌 Registros nuevos (excluidos de validaciones estrictas): 0
Se exportó correctamente el archivo de incidencias de OSLR (0 registro(s))
Los registros con errores en fechas son los siguientes:

CLAIM NUMBER: 101050/2026
  DATE OF LOSS          : 2025-08-19 00:00:00
  DATE OF REPORT        : 2025-03-02 00:00:00
  POLICY PERIOD START   : 2025-02-20 00:00:00
  POLICY PERIOD END     : 2027-02-20 00:00:00
  STATUS                : T


CLAIM NUMBER: 323361640000029
  DATE OF LOSS          : 2023-03-08 00:00:00
  DATE OF REPORT        : 2023-05-30 00:00:00
  POLICY PERIOD START   : 2021-02-20 00:00:00
  POLICY PERIOD END     : 2023-02-20 00:00:00
  STATUS                : P

Se exportó correctamente el archivo de incidencias de fechas


## 7.5 Validación empírica de OSLR (patrón histórico open/closed)

- **OSLR > 0** casi siempre (98%) corresponde a un siniestro **aún no pagado**
  (`Total Claims Paid Inward == 0`) y **reportado recientemente** (≤ 730 días).
- **OSLR = 0** domina en siniestros ya pagados y/o antiguos.
- Precisión de la heurística contra los datos reales: **97.7%**.

Marca como "atípicos" los registros cuyo OSLR calculado (Sección 6) no sigue este
patrón histórico, para revisión manual — no implica que estén mal calculados.

In [180]:
# =============================================================================
# Section 7.5: Validación empírica de OSLR (patrón histórico open/closed)
# =============================================================================
# Regla inferida estadísticamente (ver oslr_rules.py):
#   oslr_expected_status = 'open'   si Total Claims Paid Inward == 0
#                                       y días desde reporte <= DIAS_UMBRAL_OSLR_ABIERTO
#                         = 'closed' en cualquier otro caso

print('\n' + '='*95)
print('SECCIÓN 7.5: VALIDACIÓN EMPÍRICA DE OSLR (PATRÓN OPEN/CLOSED)')
print('='*95)

DIAS_UMBRAL_OSLR_ABIERTO = 730
fecha_corte = date_start_AñoMes + pd.offsets.MonthEnd(0)

def oslr_expected_status(paid_inward: float, days_since_report: float,
                          days_threshold: int = DIAS_UMBRAL_OSLR_ABIERTO) -> str:
    """Heurística empírica (no oficial): reproduce el patrón observado en datos históricos."""
    if pd.isna(days_since_report):
        return 'closed'
    if paid_inward == 0 and days_since_report <= days_threshold:
        return 'open'
    return 'closed'

df_check = df_update_db.copy()
df_check['days_since_report'] = (fecha_corte - df_check['DATE OF REPORT']).dt.days
df_check['oslr_actual_status'] = np.where(df_check[oslr_col].fillna(0) > 0, 'open', 'closed')
df_check['oslr_expected_status'] = df_check.apply(
    lambda r: oslr_expected_status(r['Total Claims Paid Inward'], r['days_since_report']),
    axis=1
)

mask_atipicos = df_check['oslr_actual_status'] != df_check['oslr_expected_status']
df_atipicos_oslr = df_check.loc[mask_atipicos, [
    'KEY LOB', 'CLAIM NUMBER', 'DATE OF REPORT', 'days_since_report',
    'Total Claims Paid Inward', oslr_col, 'oslr_actual_status', 'oslr_expected_status'
]].copy()

accuracy = 1 - mask_atipicos.mean()
print(f"Precisión de la heurística vs. OSLR real: {accuracy:.2%}")
print(f"Registros atípicos (no siguen el patrón histórico): {len(df_atipicos_oslr)}")
print("  Nota: no implica error de cálculo — solo marca registros a revisar manualmente")
if not df_check.empty:
    print(pd.crosstab(df_check['oslr_actual_status'], df_check['oslr_expected_status']))

df_atipicos_oslr.to_excel(f'{ruta_incidencias}/Incidencias_OSLR_Patron_Empirico.xlsx', index=False)
print('Se exportó correctamente el archivo de incidencias del patrón empírico de OSLR')
print('='*95 + '\n')


SECCIÓN 7.5: VALIDACIÓN EMPÍRICA DE OSLR (PATRÓN OPEN/CLOSED)
Precisión de la heurística vs. OSLR real: 96.47%
Registros atípicos (no siguen el patrón histórico): 153
  Nota: no implica error de cálculo — solo marca registros a revisar manualmente
oslr_expected_status  closed  open
oslr_actual_status                
closed                  4103   122
open                      31    81
Se exportó correctamente el archivo de incidencias del patrón empírico de OSLR



## 8. Final Adjustments

In [181]:
# =============================================================================
# Section 8: Final adjustments
# =============================================================================

print('\n' + '='*95)
print('SECCIÓN 8: AJUSTES FINALES')
print('='*95)

# Verificar que las columnas necesarias existan ANTES de hacer operaciones
print('\n🔍 Verificando columnas críticas para final adjustment:')

# Rellenar cualquier NaN restante en columnas críticas
critical_cols = [
    f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}',
    'LoB-Inward BDX', 'CLAIM NUMBER ORIGINAL',
    'Cumulative SALVAGE', 'Cumulative EXPENSES PAID', 'Cumulative VALUATION EXPENSES'
]

for col in critical_cols:
    if col in df_update_db.columns:
        nulos = df_update_db[col].isna().sum()
        if nulos > 0:
            # Rellenar NaN con 0 para columnas numéricas, vacío para texto
            if df_update_db[col].dtype in ['float64', 'int64']:
                df_update_db[col] = df_update_db[col].fillna(0)
            else:
                df_update_db[col] = df_update_db[col].fillna('')
            print(f"  ✓ {col}: {nulos} valores rellenados")

df_update_db.drop(columns=['LoB-Inward','CLAIM NUMBER'], inplace= True)
df_update_db.rename(columns={'LoB-Inward BDX': 'LoB-Inward','CLAIM NUMBER ORIGINAL' :'CLAIM NUMBER',
                             f'OSLR Inward {AñoMes}': 'OSLR Inward',
                             f'GROSS RESERVE {AñoMes}': f'GROSS RESERVE',
                             f'DEDUCTIBLE {AñoMes}': f'DEDUCTIBLE'
                             }, inplace= True)

# Rename of  policies 
df_update_db.loc[df_update_db['INWARD POLICY N°'] == '90600-323484','INWARD POLICY N°'] = '90600 323484'
df_update_db.loc[df_update_db['INWARD POLICY N°'] == '90600-328256','INWARD POLICY N°'] = '90600 328256'

#LoB normalization
# LoB of legacy info
df_update_db.loc[df_update_db['LoB-Inward'] == 'CASCO', 'LoB-Inward'] = 'CASCO Y MAQUINARIA'

# Subsidiary Normalization
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].str.strip() # Delete all the spaces at the end and beggining

# Normalize the 'SUBSIDIARY' column
df_update_db["SUBSIDIARY"] = (
df_update_db["SUBSIDIARY"]
    #todo en mayusculas
    .str.upper()
    #remueve la palabra pemex
    .str.replace(r"\bPEMEX\b", "", regex=True)
    #remueve espacios extras
    .str.replace(r"\s+", " ", regex=True)
    #elimina espacios al inicio y final
    .str.strip()
)
# Initialize a dictionary with the correct names
dict_subsidiaries = {
    'PEMEX Logistica':'LOGÍSTICA',
    'LOGISTICA':'LOGÍSTICA',
    'Logistica':'LOGÍSTICA',
    'Pemex Logística':'LOGÍSTICA',
    'LOG':'LOGÍSTICA'
}
# Apply the correction 
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].replace(dict_subsidiaries)

#Cambiamos los valores de la columna Subsidiary
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].replace({
'LOGÍSTICA':" DIRECCIÓN DE LOGÍSTICA Y SALVAGUARDIA ESTRATÉGICA ",
'REFINACIÓN':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'PETROQUÍMICA':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA", 
'EXPLORACIÓN Y PRODUCCIÓN':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN", 
'PMI':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'ETILENO':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA", 
'PERFORACIÓN Y SERVICIOS':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'TRANSFORMACIÓN INDUSTRIAL':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'REF':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'PEP':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'PTRI':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'GAS Y PETROQUÍMICA BÁSICA':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA"})

# Fill NaN with 'NO ESPECIFICADO'
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].fillna('NO ESPECIFICADO')

#Columns to drop
# Columnas fijas que siempre se eliminan
drop_fixed = [
    'KEY DED', 'CLAIM NUMBER ORIGINAL BDX',
    'POLICY PERIOD START DATE BDX', 'POLICY PERIOD END DATE BDX',
    'Monthly', 'CEDANT OBSERVATIONS', 'KOT OBSERVATIONS', 'DIF OSLR'
]
# Columnas con sufijo de período (YYYYMM) que quedan tras el procesamiento
drop_period = [col for col in df_update_db.columns if re.search(r'\s\d{6}$', col)]
dropcolumns = drop_fixed + drop_period

# Final order
list_columns_order = [
'YEAR OF THE RESERVE','ACCIDENT YEAR','CLAIMS-ID','CLAIM NUMBER','DATE OF LOSS','DATE OF REPORT','ATTRITIONAL/LARGE',	
'INWARD POLICY','INWARD POLICY N°','POLICY PERIOD START DATE','POLICY PERIOD END DATE','SUBSIDIARY','StandRe LoB',
'StandRe detailed LoB','LoB-Inward','COVER','COVERAGE','STATUS','REF. PEMEX','LOCATION','DESCRIPTION' ,
'Cumulative SALVAGE','Cumulative EXPENSES PAID','Cumulative VALUATION EXPENSES','Cumulative CLAIMS PAID',
'Total Claims Paid Inward','Total Claims Paid no Alae','OSLR Inward','Inward Incurred',
'Inward Incurred no Alae',f'GROSS RESERVE',f'DEDUCTIBLE','KOT SHARE','UW Year','Nat Cat', 'KEY LOB','FLAG CONTA','NOTAS'
] #,'DIF OSLR', 'FLAG CONTA'

# Filtrar solo las columnas que existen en el dataframe
list_columns_order_final = [col for col in list_columns_order if col in df_update_db.columns]

# Agregar las columnas restantes que no estén en el orden especificado
cols_faltantes = [col for col in df_update_db.columns if col not in list_columns_order_final]
list_columns_order_final.extend(cols_faltantes)

print(f'\n✓ Reordenando columnas (Orden final: {len(list_columns_order)} columnas)')
df_update_db = df_update_db[list_columns_order]
print(f'✓ Nuestra base final cuenta con {len(df_update_db)} registros')

# =============================================================================
# GUARDIA DE PROTECCION LEGACY (paso 2/2): re-validar tras rename/fillna/reorder
# =============================================================================
# Compara contra la foto tomada al final de la Seccion 6 (df_legacy_guard_snapshot).
# Si CUALQUIER cambio futuro en el codigo de la Seccion 7 u 8 llegara a tocar
# Reserva/Deducible/OSLR de una poliza legacy, esto detiene el proceso en vez
# de dejarlo pasar silenciosamente.
# Se compara por el indice original (ver nota en la Seccion 6) en vez de por
# KEY LOB, para que un multi-deducible legitimo no produzca falsos positivos.
if not df_legacy_guard_snapshot.index.isin(df_update_db.index).all():
    raise ValueError(
        "GUARDIA LEGACY (nb2): algunos registros legacy desaparecieron de df_update_db "
        "entre la Seccion 6 y la Seccion 8. Revisar antes de exportar."
    )

_check_legacy = df_update_db.loc[df_legacy_guard_snapshot.index, ['GROSS RESERVE', 'DEDUCTIBLE', 'OSLR Inward']]

_mismatch_legacy = (
    (_check_legacy['GROSS RESERVE'].round(2).to_numpy() != df_legacy_guard_snapshot['GROSS RESERVE'].round(2).to_numpy()) |
    (_check_legacy['DEDUCTIBLE'].round(2).to_numpy() != df_legacy_guard_snapshot['DEDUCTIBLE'].round(2).to_numpy()) |
    (_check_legacy['OSLR Inward'].fillna(0).round(2).to_numpy() != df_legacy_guard_snapshot['OSLR Inward'].fillna(0).round(2).to_numpy())
)
if _mismatch_legacy.any():
    raise ValueError(
        f"GUARDIA LEGACY (nb2): {int(_mismatch_legacy.sum())} registro(s) legacy cambiaron de valor "
        "en Reserva/Deducible/OSLR entre la Seccion 6 y la Seccion 8. Revisar antes de exportar."
    )
print(f"\u2705 Guardia legacy (paso 2/2): {len(df_legacy_guard_snapshot)} registros verificados sin cambios antes de exportar.")
del _check_legacy, _mismatch_legacy




SECCIÓN 8: AJUSTES FINALES

🔍 Verificando columnas críticas para final adjustment:
  ✓ LoB-Inward BDX: 984 valores rellenados

✓ Reordenando columnas (Orden final: 38 columnas)
✓ Nuestra base final cuenta con 4337 registros
✅ Guardia legacy (paso 2/2): 937 registros verificados sin cambios antes de exportar.


In [182]:
# =============================================================================
#  Validación de pagos antes de exportar
# =============================================================================
def validar_pagos_antes_exportacion(
    df_conta_final, df_update_db,
    key_col="KEY LOB", claim_col="CLAIMS-ID"
):
    """
    Valida que todo lo que está en df_conta_final
    exista en df_update_db, tanto por KEY LOB como por CLAIMS-ID.
    Detiene el código si hay faltantes.
    """
    errores = []

    # ── Validación 1: KEY LOB ──────────────────────────────────────────
    keys_conta = set(df_conta_final[key_col])
    keys_db    = set(df_update_db[key_col])
    faltantes_key = keys_conta - keys_db

    if faltantes_key:
        df_f = df_conta_final[
            df_conta_final[key_col].isin(faltantes_key)
        ][[key_col]].drop_duplicates()
        errores.append(
            f"[1] KEY LOB de df_conta_final no encontrados en df_update_db: {len(df_f)}\n"
            + df_f.to_string(index=False)
        )

    # ── Resultado ─────────────────────────────────────────────────────
    if errores:
        sep = "=" * 55
        detalle = ("\n" + "─" * 55 + "\n").join(errores)
        raise ValueError(
            f"\n{sep}\n"
            f"  EXPORTACIÓN DETENIDA — {len(errores)} problema(s)\n"
            f"{sep}\n"
            f"{detalle}\n"
            f"{sep}\n"
            f"  Corrige los errores antes de exportar."
        )

    print("✓ Validación OK — todos los pagos de df_conta_final están en df_update_db.")
# ── paso 6: validar ANTES de exportar ─────────────────
validar_pagos_antes_exportacion(
    df_conta_final = df_conta_final,
    df_update_db   = df_update_db,
)

#Redondear columnas numéricas a 2 decimales
df_update_db = df_update_db.round(2)

✓ Validación OK — todos los pagos de df_conta_final están en df_update_db.


C:\Users\IKAL14\AppData\Local\Temp\ipykernel_49192\755079885.py:50: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_update_db = df_update_db.round(2)


In [183]:
# =============================================================================
# RESUMEN DE DIFERENCIAS OSLR vs BASE MANUAL — masivo, con causa por siniestro
# Pegar como celda nueva justo DESPUÉS de la Sección 10 (Validación OSLR vs BDX)
# de "2.-Actualizacion Contable Marine.ipynb".
# =============================================================================
import os
import re
import shutil
import tempfile
import time
import pandas as pd  # import idempotente: no rompe nada si ya estaba importado


def _texto_seguro(serie):
    """
    Convierte una Serie a texto elemento por elemento con el str() nativo de
    Python, en vez de la versión vectorizada .astype(str).str.strip(). Esta
    última puede fallar o comportarse distinto según el dtype de origen
    (categórico, el dtype 'string' que pandas 2/3 puede asignar por defecto
    a columnas leídas de Excel, columnas con NaN mezclado, etc.) y fue la
    causa de un TypeError ("expected str instance, float found") al armar
    STATUS. Con .map(str) elemento a elemento el resultado es siempre un
    string real, sin importar la versión de pandas ni el dtype original.
    Los NaN/None quedan como cadena vacía en vez de la palabra 'nan'.
    """
    return serie.map(lambda v: '' if pd.isna(v) else str(v).strip())


def _normalizar_claim_number(serie):
    """
    Normaliza una columna de Claim Number a texto ANTES de pasarla por
    _limpiar_llave. Soluciona el caso muy común en Excel de que la columna
    venga numérica (int/float) y, si hay aunque sea una celda vacía en la
    columna, pandas la sube a float64 -> "123" se lee como 123.0 y al hacer
    str() queda "123.0", que ya no calza contra el "123" (texto) que trae
    df_update_db. Sin este paso, TODOS los siniestros salían como "Sin match"
    aunque estuvieran en ambas bases.
    """
    s = serie.copy()
    if pd.api.types.is_numeric_dtype(s):
        return s.apply(lambda v: '' if pd.isna(v) else str(int(v)) if float(v).is_integer() else str(v))
    # Por si la columna es texto pero algún proceso previo (ej. un CSV
    # intermedio) ya la dejó como "123.0": se le quita el ".0" sobrante,
    # sin tocar valores que sí tienen decimales reales.
    return _texto_seguro(s).str.replace(r'^(-?\d+)\.0$', r'\1', regex=True)


def _a_numero_oslr(serie):
    """
    Convierte la columna OSLR Inward de la base manual a número, tolerando
    formatos típicos de Excel: separador de miles (1,500.00), signo $,
    espacios y negativos entre paréntesis (1,500.00). Antes, pd.to_numeric
    con errors='coerce' + fillna(0) convertía CUALQUIER valor con esos
    formatos en 0 silenciosamente, generando diferencias falsas clasificadas
    como "Sin explicar" que en realidad eran solo un problema de formato.
    Además, si después de limpiar queda algún valor que de verdad no se pudo
    interpretar como número, se avisa por consola en vez de tragárselo como 0.
    """
    texto = _texto_seguro(serie)
    es_negativo_parentesis = texto.str.match(r'^\(.*\)$')
    limpio = (
        texto.str.replace(r'[()]', '', regex=True)
        .str.replace(r'[^0-9.\-]', '', regex=True)
    )
    numero = pd.to_numeric(limpio, errors='coerce')
    numero = numero.mask(es_negativo_parentesis, -numero.abs())
    valores_no_parseados = serie[
        numero.isna() & ~texto.isin(['', 'nan', 'None', 'NaT', 'nat'])
    ]
    if not valores_no_parseados.empty:
        ejemplos = list(pd.unique(valores_no_parseados))[:10]
        print(
            f"  ⚠️ {len(valores_no_parseados)} valor(es) de 'OSLR Inward' en la base manual "
            f"no se pudieron leer como número y se tomaron como 0. Revisa: {ejemplos}"
        )
    return numero.fillna(0)


def resumen_diferencias_oslr_vs_manual(
    path_manual_excel, df_update_db, LEGACY, df_log_zereo_oslr, df_ajustes_net_reserve,
    ruta_incidencias, AñoMes, sheet_name=0, tolerancia=1.0,
):
    """
    Compara el OSLR final calculado por el notebook (df_update_db['OSLR Inward'],
    agregado por CLAIM NUMBER) contra una base manual (Excel externo, agregado
    igual por CLAIM NUMBER), y para cada siniestro cuya diferencia supera la
    tolerancia, clasifica la causa más probable reutilizando lo que ya calculó
    la Sección 6: reglas de zereo (df_log_zereo_oslr), ajuste contra Net Reserve
    (df_ajustes_net_reserve) y pólizas LEGACY.

    Retorna
    -------
    pd.DataFrame con columnas CLAIM NUMBER, INWARD POLICY N°, STATUS,
    Filas KEY LOB, OSLR Manual, OSLR Calculado, Diferencia, Causa, Detalle.
    None si falta el archivo, faltan columnas obligatorias, o hay un error
    inesperado (se informa por consola y no se interrumpe el resto del notebook).
    """
    tmp_dir = None
    try:
        if not os.path.exists(path_manual_excel):
            raise FileNotFoundError(f"No se encontró la base manual en: {path_manual_excel}")

        tmp_dir = tempfile.mkdtemp(prefix='oslr_manual_')
        path_tmp = os.path.join(tmp_dir, os.path.basename(path_manual_excel))
        ultimo_error = None
        for intento in range(5):
            try:
                shutil.copy2(path_manual_excel, path_tmp)
                ultimo_error = None
                break
            except PermissionError as e:
                ultimo_error = e
                print(f"  ⏳ Archivo bloqueado (probablemente sincronizando en OneDrive), reintento {intento + 1}/5...")
                time.sleep(3)
        if ultimo_error is not None:
            raise PermissionError(
                f"No se pudo leer '{path_manual_excel}' tras varios intentos: {ultimo_error}. "
                "Verifica en el icono de OneDrive que el archivo terminó de sincronizar (nube/check verde) e intenta de nuevo."
            )

        df_manual_raw = pd.read_excel(path_tmp, sheet_name=sheet_name)
        faltantes_manual = {'CLAIM NUMBER', 'OSLR Inward'} - set(df_manual_raw.columns)
        if faltantes_manual:
            raise KeyError(f"La base manual no trae las columnas esperadas: {faltantes_manual}")
        faltantes_calc = {'CLAIM NUMBER', 'INWARD POLICY N°', 'STATUS', 'OSLR Inward'} - set(df_update_db.columns)
        if faltantes_calc:
            raise KeyError(
                f"df_update_db no trae las columnas esperadas: {faltantes_calc}. "
                "Corre esta celda DESPUÉS de la Sección 8 (rename de OSLR Inward)."
            )
        if LEGACY is None:
            LEGACY = []

        print('\n' + '=' * 95)
        print(f'RESUMEN DE DIFERENCIAS OSLR vs BASE MANUAL — {AñoMes}')
        print('=' * 95)

        # --- Normalización de llave: primero números "sucios" -> texto limpio,
        # luego la misma lógica que _limpiar_llave de la Sección 10 ---
        limpiar = _limpiar_llave if '_limpiar_llave' in globals() else _texto_seguro

        df_manual = df_manual_raw.copy()
        df_manual['CLAIM NUMBER'] = limpiar(_normalizar_claim_number(df_manual['CLAIM NUMBER']))
        df_manual['OSLR Inward'] = _a_numero_oslr(df_manual['OSLR Inward'])
        manual_por_claim = (
            df_manual.groupby('CLAIM NUMBER', as_index=False)['OSLR Inward']
            .sum()
            .rename(columns={'OSLR Inward': 'OSLR Manual'})
        )

        df_calc = df_update_db.copy()
        df_calc['CLAIM NUMBER'] = limpiar(_normalizar_claim_number(df_calc['CLAIM NUMBER']))
        df_calc['INWARD POLICY N°'] = _texto_seguro(df_calc['INWARD POLICY N°'])
        calc_por_claim = df_calc.groupby('CLAIM NUMBER', as_index=False).agg(**{
            'OSLR Calculado': ('OSLR Inward', 'sum'),
            'INWARD POLICY N°': ('INWARD POLICY N°', 'first'),
            'STATUS': ('STATUS', lambda s: ', '.join(sorted(set(v for v in _texto_seguro(s) if v)))),
            'Filas KEY LOB': ('CLAIM NUMBER', 'count'),
        })

        resumen = manual_por_claim.merge(calc_por_claim, on='CLAIM NUMBER', how='outer')
        resumen['OSLR Manual'] = resumen['OSLR Manual'].fillna(0)
        resumen['OSLR Calculado'] = resumen['OSLR Calculado'].fillna(0)
        resumen['Diferencia'] = resumen['OSLR Manual'] - resumen['OSLR Calculado']
        resumen = resumen[resumen['Diferencia'].abs() > tolerancia].copy()

        if resumen.empty:
            print(f'✅ Ningún siniestro supera la tolerancia (${tolerancia:,.2f}). Bases conciliadas.')
            return resumen

        # Columnas que quedan NaN cuando el claim solo existe en una de las dos
        # bases (después del merge outer) -> mejor vacío legible que NaN en el Excel.
        resumen['INWARD POLICY N°'] = resumen['INWARD POLICY N°'].fillna('(sin match)')
        resumen['STATUS'] = resumen['STATUS'].fillna('(sin match)')
        resumen['Filas KEY LOB'] = resumen['Filas KEY LOB'].fillna(0).astype(int)

        # --- Contexto ya calculado en la Sección 6, para clasificar la causa ---
        reglas_por_claim = {}
        if df_log_zereo_oslr is not None and not df_log_zereo_oslr.empty and 'CLAIM NUMBER' in df_log_zereo_oslr.columns and 'REGLA' in df_log_zereo_oslr.columns:
            reglas_por_claim = (
                df_log_zereo_oslr.assign(**{'CLAIM NUMBER': limpiar(_normalizar_claim_number(df_log_zereo_oslr['CLAIM NUMBER']))})
                .groupby('CLAIM NUMBER')['REGLA']
                .apply(lambda s: ', '.join(sorted(set(v for v in _texto_seguro(s) if v))))
                .to_dict()
            )

        claims_ajuste_net_reserve = set()
        if df_ajustes_net_reserve is not None and not df_ajustes_net_reserve.empty and 'CLAIM NUMBER' in df_ajustes_net_reserve.columns:
            claims_ajuste_net_reserve = set(limpiar(_normalizar_claim_number(df_ajustes_net_reserve['CLAIM NUMBER'])))

        claims_en_manual = set(manual_por_claim['CLAIM NUMBER'])
        claims_en_calc = set(calc_por_claim['CLAIM NUMBER'])
        legacy_set = set(str(p).strip() for p in (LEGACY or []))

        def _clasificar(row):
            claim = row['CLAIM NUMBER']
            if claim not in claims_en_calc:
                return 'Sin match en base calculada', 'El Claim Number no está en df_update_db (¿nuevo, no cargado, o formato distinto en la base manual?).'
            if claim not in claims_en_manual:
                return 'Sin match en base manual', 'El Claim Number no está en la base manual (¿pendiente de captura, o formato distinto?).'
            if str(row['INWARD POLICY N°']).strip() in legacy_set:
                return 'Legacy', 'Póliza LEGACY: el notebook congela el OSLR (0 / valor del mes anterior) por diseño; la base manual puede traer otro valor si no se actualizó con el mismo criterio.'
            if claim in reglas_por_claim:
                regla = reglas_por_claim[claim]
                nota_base_anterior = ' (usa OSLR/variables del mes anterior)' if re.search(r'anterior|previo|cambios en el mes', regla, re.IGNORECASE) else ''
                return f'Zereado por regla: {regla}', f'Ver Log_Zereo_OSLR.xlsx para el detalle{nota_base_anterior}.'
            if claim in claims_ajuste_net_reserve:
                return 'Ajuste vs Net Reserve BDX', 'El OSLR se forzó a Net Reserve (USD) del BDX (paso 2 de la lógica de OSLR, STATUS=P y Net Reserve≈0).'
            return 'Sin explicar', 'Ninguna regla conocida movió este siniestro: revisar cálculo (GROSS RESERVE/DEDUCTIBLE/Cumulative CLAIMS PAID) o el dato de origen en ambas bases.'

        causas = resumen.apply(_clasificar, axis=1, result_type='expand')
        causas.columns = ['Causa', 'Detalle']
        resumen[['Causa', 'Detalle']] = causas

        resumen = resumen.sort_values(by='Diferencia', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
        resumen = resumen[[
            'CLAIM NUMBER', 'INWARD POLICY N°', 'STATUS', 'Filas KEY LOB',
            'OSLR Manual', 'OSLR Calculado', 'Diferencia', 'Causa', 'Detalle',
        ]]

        print(f"\nSiniestros con diferencia > ${tolerancia:,.2f}: {len(resumen)}")
        print(f"Diferencia neta (Manual - Calculado): ${resumen['Diferencia'].sum():,.2f}")
        print('\nResumen por causa (cantidad de siniestros y $ absoluto):')
        resumen_causa = resumen.groupby('Causa').agg(
            **{'N Siniestros': ('CLAIM NUMBER', 'count'), '$ Diferencia (abs)': ('Diferencia', lambda s: s.abs().sum())}
        ).sort_values('$ Diferencia (abs)', ascending=False)
        print(resumen_causa.to_string())

        path_out = f'{ruta_incidencias}/Resumen_Diferencias_OSLR_vs_Manual_{AñoMes}.xlsx'
        resumen.to_excel(path_out, index=False)
        print(f"\n📄 Exportado: {path_out}")
        return resumen

    except FileNotFoundError as e:
        print(f"❌ {e}")
        return None
    except PermissionError as e:
        print(f"❌ {e}")
        return None
    except KeyError as e:
        print(f"❌ Falta una columna esperada: {e}")
        print("   (si el mensaje no menciona 'CLAIM NUMBER' ni 'OSLR Inward', revisa también "
              "'REGLA' en df_log_zereo_oslr y 'CLAIM NUMBER' en df_ajustes_net_reserve)")
        return None
    except Exception as e:
        import traceback
        print(f"❌ Error inesperado al armar el resumen de diferencias: {e}")
        traceback.print_exc()
        return None
    finally:
        if tmp_dir is not None:
            shutil.rmtree(tmp_dir, ignore_errors=True)


# ------------------------- Ejecución -------------------------
# Ajusta la ruta de la base manual a la del período que estés validando.
_requeridas = ['df_update_db', 'LEGACY', 'ruta_incidencias', 'AñoMes']
_faltantes = [v for v in _requeridas if v not in globals()]
if _faltantes:
    print(f"❌ Faltan variables en memoria: {_faltantes}. Corre primero las Secciones 2, 6, 8 y 10 del notebook antes de esta celda.")
else:
    PATH_MANUAL_OSLR = rf"C:\Users\IKAL14\OneDrive - Kot Insurance Company AG\Transporte, Carga y Embarcaciones\{AñoMes}_Siniestros_Marine_MANUAL.xlsx"
    df_resumen_diferencias_manual = resumen_diferencias_oslr_vs_manual(
        path_manual_excel=PATH_MANUAL_OSLR,
        df_update_db=df_update_db,
        LEGACY=LEGACY,
        df_log_zereo_oslr=df_log_zereo_oslr if 'df_log_zereo_oslr' in globals() else None,
        df_ajustes_net_reserve=df_ajustes_net_reserve if 'df_ajustes_net_reserve' in globals() else None,
        ruta_incidencias=ruta_incidencias,
        AñoMes=AñoMes,
    )


RESUMEN DE DIFERENCIAS OSLR vs BASE MANUAL — 202604
✅ Ningún siniestro supera la tolerancia ($1.00). Bases conciliadas.


## 9. Final Export

In [184]:
# =============================================================================
# Section 9: Final Export
# =============================================================================

# Final database after merge
if not os.path.exists(f'{ruta_procesados}/Final'):
    os.makedirs(f'{ruta_procesados}/Final')

df_update_db.to_excel(f'{ruta_procesados}/Final/{AñoMes}_Siniestros_Marine.xlsx', index=False)
df_update_db.to_pickle(f'{ruta_procesados}/Final/{AñoMes}_Siniestros_Marine.pkl')

## 10. Validación OSLR vs BDX (+ Log Zereo)

Corre la validación policy-level automáticamente al final de este mismo notebook,
usando los DataFrames que ya están en memoria (`df_update_db`, `df_log_zereo_oslr`)
en vez de volver a leer los exports de este notebook desde disco. El único archivo
externo que se lee es `OSLR {AñoMes}.xlsx` (output de `OSLR.ipynb`), que es la
única fuente que este notebook no calcula por sí mismo.

Genera `Validacion_OSLR_{AñoMes}.xlsx` en la carpeta de Incidencias del mes, con
4 hojas: Validación, Detalle BDX, Log Zereo OSLR, Zereos por Póliza. Si falta el
archivo de BDX o las columnas esperadas, se avisa por consola y el resto del
notebook (ya guardado en la Sección 9) no se ve afectado.

In [185]:
# =============================================================================
# Sección 10: Validación automática OSLR vs BDX (+ cruce con Log Zereo OSLR,
#              + detalle por Siniestro)
# =============================================================================
# Los archivos externos que se leen son:
#   - 'OSLR {AñoMes}.xlsx' (output de OSLR.ipynb, en route_account) — BDX a
#     nivel póliza, usado SOLO en la pestaña Validación.
#   - 'Payments_{AñoMes}.xlsx' (archivo contable, en route_account, ya usado
#     en la Sección 3) — Outstanding por claim.
#   - '{AñoMes}_BDX_Marine.xlsx' (en Marine/Procesados/{AñoMes}, la misma carpeta
#     raíz que route_validations pero reemplazando 'Validaciones' por
#     'Procesados') — BDX Net Reserve (USD) REAL por Claim Number, usado en
#     Siniestro y Outstanding Payments.
#
# Si falta alguno de esos archivos o le faltan columnas, se avisa por consola
# y se continúa sin detener el kernel: lo ya exportado en la Sección 9 no se
# ve afectado por un fallo aquí.
#
# =============================================================================

# --- Config: nombres de columnas del archivo Payments_{AñoMes}.xlsx ---------
# Ajusta estos 4 nombres si no coinciden con las columnas reales del archivo.
# PAYMENTS_COL_STATUS es la columna cuyo VALOR filtramos (nos quedamos con
# las filas donde valga "Outstanding"); PAYMENTS_COL_AMOUNT es la columna
# numérica que se suma para esas filas — son cosas distintas. PAYMENTS_COL_POLICY
# se usa SOLO para los claims Outstanding que no existen en Reserve Database
# (df_update_db), para poder mostrarlos igual en la hoja Outstanding Payments
# con su Policy Number en vez de descartarlos.
PAYMENTS_COL_CLAIM = 'Claims Reference'
PAYMENTS_COL_STATUS = 'Status'
PAYMENTS_COL_AMOUNT = 'Amount USD (CORRECT)'
PAYMENTS_COL_POLICY = 'Policy N°'

# --- Config: renombres de Policy Number para la hoja Siniestro/Outstanding --
# Distintas fuentes usan formatos distintos para la MISMA póliza (más allá
# de espacios, que ya arregla _limpiar_llave). Este mapping normaliza esos
# casos a un solo nombre canónico antes de cruzar Reserve Database, BDX real
# y Payments, para que no aparezcan como pólizas separadas. Agrega más
# entradas aquí según se detecten (llave = tal como sale de Reserve
# Database / df_update_db, ya sin espacios; valor = el nombre canónico que
# se quiere ver en el reporte).
POLICY_NUMBER_RENAME = {
    'E01-2-60-000000010_0000-0-1': 'E01-2-60-10',
    'E01-2-60-000000003_0000-0-1': 'E01-2-60-3',
}


def _limpiar_llave(serie):
    '''
    Normaliza una columna que se va a usar como llave de cruce (Claim Number
    o Policy Number en la hoja Siniestro/Outstanding Payments): la pasa a
    str y le quita TODOS los espacios (no solo los de los extremos, sino
    también los internos, incluyendo alrededor de separadores como "/").

    Se necesita porque el mismo claim/póliza puede venir formateado distinto
    entre Reserve Database, el BDX real por claim ({AñoMes}_BDX_Marine.xlsx) y
    Payments — ej. "35100 3041576" vs "351003041576", o "4900 / 2023" vs
    "4900/2023". Sin esta limpieza, el merge por Claim Number no los
    reconoce como el mismo claim y aparecen como dos filas separadas, cada
    una "explicando" a la otra con una diferencia falsa (mismo monto, un
    lado con Reserve Database y el otro con BDX Reserve).

    Solo se usa para las funciones de esta sección que arman las hojas
    Siniestro / Outstanding Payments (_construir_detalle_claims y sus
    fuentes); NO se aplica a la pestaña Validación, que sigue comparando
    Policy Number tal cual vienen en OSLR {AñoMes}.xlsx y df_update_db (ahí
    no se ha visto este problema).

    Parámetros
    ----------
    serie : pandas.Series
        Columna a normalizar (Claim Number o Policy Number).

    Retorna
    -------
    pandas.Series
        Serie como str, sin ningún espacio en blanco.
    '''
    return serie.astype(str).str.replace(r'\s+', '', regex=True)


def _cargar_bdx_net_reserve_validacion(route_account, anio_mes):
    '''
    Carga el output de OSLR.ipynb (Net Reserve por póliza, extraído del BDX)
    para compararlo contra el Reserve Database calculado en este notebook.

    Parámetros
    ----------
    route_account : str
        Carpeta de contabilidad del periodo (misma variable ya usada en la
        Sección 3 para leer Payments_{AñoMes}.xlsx).
    anio_mes : int
        Periodo a validar, formato AAAAMM.

    Retorna
    -------
    pandas.DataFrame
        Columnas: Policy Number, Cedant, Cover, Net Reserve. Policy Number
        normalizado (str sin espacios), Net Reserve numérico sin NaN. Filas
        sin Policy Number/Cedant original quedan con una clave placeholder
        ("SIN POLIZA - <Cover>" / "Sin Cedant") en vez de perderse.

    Excepciones
    -----------
    FileNotFoundError
        Si no existe 'OSLR {anio_mes}.xlsx' (aún no se corrió OSLR.ipynb).
    KeyError
        Si el archivo existe pero no trae las columnas esperadas.
    '''
    path = f'{route_account}/OSLR {anio_mes}.xlsx'
    if not os.path.exists(path):
        raise FileNotFoundError(f'No se encontró el output de OSLR.ipynb para {anio_mes} en {path}. Corre OSLR.ipynb antes de validar.')
    df_bdx = pd.read_excel(path)
    faltantes = {'Policy Number', 'Cedant', 'Cover', 'Net Reserve'} - set(df_bdx.columns)
    if faltantes:
        raise KeyError(f'El output de OSLR.ipynb no trae las columnas esperadas: {faltantes}')

    df_bdx['Net Reserve'] = pd.to_numeric(df_bdx['Net Reserve'], errors='coerce').fillna(0)

    # Filas sin Policy Number (ej. reservas agregadas por Cover, tipo "D Y V")
    # no se descartan: se agrupan bajo una clave placeholder basada en Cover
    # para que el monto siga entero en el total BDX y quede visible como su
    # propia fila en la validación, en vez de perderse en el groupby.
    sin_poliza = df_bdx['Policy Number'].isna()
    if sin_poliza.any():
        monto_sin_poliza = df_bdx.loc[sin_poliza, 'Net Reserve'].sum()
        print(f'Aviso: {sin_poliza.sum()} filas sin Policy Number en el BDX de {anio_mes} '
              f'(Net Reserve total {monto_sin_poliza:,.2f}). Se agrupan bajo "SIN POLIZA - <Cover>".')
        df_bdx.loc[sin_poliza, 'Policy Number'] = 'SIN POLIZA - ' + df_bdx.loc[sin_poliza, 'Cover'].astype(str).str.strip()
        df_bdx.loc[sin_poliza, 'Cedant'] = df_bdx.loc[sin_poliza, 'Cedant'].fillna('Sin Cedant')

    df_bdx['Policy Number'] = df_bdx['Policy Number'].astype(str).str.strip()
    return df_bdx


def _cargar_bdx_reserve_por_claim_validacion(route_validations, anio_mes, tolerancia_neteo=0.01, umbral_cero=10):
    '''
    Carga {anio_mes}_BDX_Marine.xlsx (Marine/Procesados/{anio_mes}), la
    fuente con el Net Reserve (USD) REAL por Claim Number del BDX — a
    diferencia de OSLR {anio_mes}.xlsx (usado en la pestaña Validación), que
    solo trae el total por póliza. Esto reemplaza la asignación proporcional
    que se hacía antes en la hoja Siniestro/Outstanding Payments para
    "inventar" un BDX Reserve por claim a partir del total de la póliza.
    (Antes se usaba consolidado_bdx.xlsx, en la misma carpeta; se cambió a
    este archivo a pedido del usuario.)

    Antes de agregar por claim, hace dos limpiezas sobre el archivo crudo:
      1. Rellena Claim Number en filas de "corrección"/ajuste: el archivo
         trae una columna '#' con valores como 1, 1.1, 1.2, 1.3, 1.4 — la
         fila entera (1) trae el Claim Number, pero sus filas decimales
         asociadas (1.1, 1.2...) lo traen vacío aunque sí tienen su propio
         Net Reserve (USD) e INWARD POLICY N°. Se agrupan las filas por la
         parte entera de '#' y se rellena el Claim Number vacío con el de su
         grupo (ffill + bfill), para no perder esas filas al agregar por
         Claim Number. Si el archivo no trae columna '#', se avisa y se
         omite este paso (las filas con Claim Number vacío quedarían sin
         agrupar correctamente).
      2. Aplica un umbral de limpieza: cualquier |Net Reserve (USD)| menor a
         umbral_cero se convierte a 0 ANTES de sumar por claim. Esto es
         para "planchar" ruido de punto flotante que puede traer el archivo
         (ej. 5.82077E-11, 2.27374E-13, -1.81899E-12), que en la práctica
         son ceros y no reservas reales.

    Este archivo puede traer más de una fila para el mismo Claim Number (ej.
    un registro original y su reverso/corrección). Se suman TODAS las filas
    de cada claim — si dos registros se cancelan entre sí (ej. +3740.49 y
    -3740.49), la suma da ~0 y ese claim sale con BDX Reserve ~0 en vez de
    con el monto de una sola de las dos filas, evitando que aparezca como
    una diferencia falsa en la hoja Siniestro. Antes de agregar, se avisa
    por consola cuántos Claim Number tienen más de una fila y, de esos,
    cuántos netean a ~0 vs cuántos no (esos últimos si representan una
    reserva real y conviene revisarlos).

    No hay una variable propia en el notebook para la carpeta
    Marine/Procesados/{anio_mes}; se arma reemplazando 'Validaciones' por
    'Procesados' en route_validations (mismo patrón de carpeta raíz
    Marine/{Validaciones,Procesados}/{anio_mes}).

    Parámetros
    ----------
    route_validations : str
        Carpeta de validaciones del periodo (la misma variable que arma
        ruta_validacion_oslr más abajo, ej. ".../Marine/Validaciones/202510").
    anio_mes : int
        Periodo a validar, formato AAAAMM.
    tolerancia_neteo : float, default 0.01
        Debajo de este valor (USD), la suma de las filas duplicadas de un
        claim se considera "neteada a 0" para el aviso de consola (no afecta
        el cálculo, que siempre suma el total real).
    umbral_cero : float, default 10
        |Net Reserve (USD)| por debajo de este valor se convierte a 0 antes
        de sumar por claim (limpieza de ruido de punto flotante).

    Retorna
    -------
    pandas.DataFrame
        Columnas: Policy Number, Claim Number, BDX Reserve. Policy Number y
        Claim Number normalizados (str sin espacios), BDX Reserve numérico
        sin NaN, agregado (sum) por si un mismo claim trae varias filas en
        {anio_mes}_BDX_Marine.xlsx.

    Excepciones
    -----------
    ValueError
        Si route_validations no contiene 'Validaciones' (no se puede armar
        la ruta a Marine/Procesados/{anio_mes} con el patrón esperado).
    FileNotFoundError
        Si no existe {anio_mes}_BDX_Marine.xlsx para el periodo.
    KeyError
        Si el archivo existe pero no trae INWARD POLICY N°, CLAIM NUMBER o
        Net Reserve (USD).
    '''
    if 'Validaciones' not in route_validations:
        raise ValueError(
            f'No se pudo armar la ruta a Marine/Procesados/{anio_mes}: route_validations '
            f'("{route_validations}") no contiene "Validaciones". Ajusta esta función si la '
            f'carpeta de {anio_mes}_BDX_Marine.xlsx no sigue ese mismo patrón de carpeta raíz.'
        )
    route_procesados = route_validations.replace('Validaciones', 'Procesados')
    path = f'{route_procesados}/{anio_mes}_BDX_Marine.xlsx'
    if not os.path.exists(path):
        raise FileNotFoundError(f'No se encontró {anio_mes}_BDX_Marine.xlsx para {anio_mes} en {path}.')

    df_bdx_claim = pd.read_excel(path)
    faltantes = {'INWARD POLICY N°', 'CLAIM NUMBER', 'Net Reserve (USD)'} - set(df_bdx_claim.columns)
    if faltantes:
        raise KeyError(f'{anio_mes}_BDX_Marine.xlsx no trae las columnas esperadas: {faltantes}')

    # 1. Rellenar Claim Number en filas de corrección (mismo '#' con
    # decimales, ej. 1, 1.1, 1.2, 1.3, 1.4 -> todas deben quedar con el
    # Claim Number de la fila entera "1"). IMPORTANTE: la columna '#' se
    # reinicia en 1 para cada BDX_SOURCE_FILE/BDX_SOURCE_SHEET (cada hoja de
    # origen tiene su propio 1, 2, 3...), así que agrupar SOLO por '#' uniría
    # por error filas de siniestros no relacionados de hojas distintas que
    # comparten el mismo número. Se agrupa por (BDX_SOURCE_FILE,
    # BDX_SOURCE_SHEET, parte entera de '#') cuando esas columnas existen.
    if '#' in df_bdx_claim.columns:
        grupo_num = pd.to_numeric(df_bdx_claim['#'], errors='coerce').apply(np.floor)
        cols_bloque = [c for c in ('BDX_SOURCE_FILE', 'BDX_SOURCE_SHEET') if c in df_bdx_claim.columns]
        if cols_bloque:
            claves_grupo = [df_bdx_claim[c] for c in cols_bloque] + [grupo_num]
        else:
            claves_grupo = [grupo_num]
            print(f'Aviso: {anio_mes}_BDX_Marine.xlsx no trae BDX_SOURCE_FILE/BDX_SOURCE_SHEET; se agrupa '
                  "solo por la columna '#', que puede repetirse entre distintas hojas de origen y mezclar "
                  'Claim Number de siniestros no relacionados. Revisar el resultado con cuidado.')

        df_bdx_claim['CLAIM NUMBER'] = df_bdx_claim['CLAIM NUMBER'].replace(r'^\s*$', np.nan, regex=True)
        n_vacios_antes = df_bdx_claim['CLAIM NUMBER'].isna().sum()
        df_bdx_claim['CLAIM NUMBER'] = (
            df_bdx_claim.groupby(claves_grupo, dropna=False)['CLAIM NUMBER']
            .transform(lambda s: s.ffill().bfill())
        )
        n_rellenados = n_vacios_antes - df_bdx_claim['CLAIM NUMBER'].isna().sum()
        if n_rellenados:
            print(f'Aviso: se rellenaron {n_rellenados} Claim Number vacíos en {anio_mes}_BDX_Marine.xlsx '
                  "usando el Claim Number de su fila entera en la columna '#' (ej. 1.1/1.2 -> el de la fila 1).")
    else:
        print(f"Aviso: {anio_mes}_BDX_Marine.xlsx no trae columna '#'; no se pudo rellenar Claim Number en "
              'filas de corrección (si las hay, van a quedar sin Claim Number y se perderán del cruce por claim).')

    df_bdx_claim['Net Reserve (USD)'] = pd.to_numeric(df_bdx_claim['Net Reserve (USD)'], errors='coerce').fillna(0)

    # 2. Umbral: ruido de punto flotante (ej. 5.82077E-11) se convierte a 0
    # antes de sumar, para que no se confunda con reserva real.
    bajo_umbral = df_bdx_claim['Net Reserve (USD)'].abs() < umbral_cero
    if bajo_umbral.any():
        print(f'Aviso: {bajo_umbral.sum()} filas en {anio_mes}_BDX_Marine.xlsx con |Net Reserve (USD)| < '
              f'{umbral_cero} se convirtieron a 0 (ruido de punto flotante).')
        df_bdx_claim.loc[bajo_umbral, 'Net Reserve (USD)'] = 0.0

    df_bdx_claim['CLAIM NUMBER'] = _limpiar_llave(df_bdx_claim['CLAIM NUMBER'])
    df_bdx_claim['INWARD POLICY N°'] = _limpiar_llave(df_bdx_claim['INWARD POLICY N°']).replace(POLICY_NUMBER_RENAME)

    # Aviso de transparencia: claims con más de una fila en este archivo, y
    # cuántos de esos netean a ~0 al sumarlos (no generan diferencia falsa)
    # vs cuántos no (reserva real repartida en varias filas, revisar).
    n_filas_por_claim = df_bdx_claim.groupby('CLAIM NUMBER')['CLAIM NUMBER'].transform('count')
    duplicados = df_bdx_claim[n_filas_por_claim > 1]
    if not duplicados.empty:
        suma_por_claim = duplicados.groupby('CLAIM NUMBER')['Net Reserve (USD)'].sum()
        n_netean_cero = (suma_por_claim.abs() <= tolerancia_neteo).sum()
        n_no_netean = len(suma_por_claim) - n_netean_cero
        print(f'Aviso: {len(suma_por_claim)} Claim Number en {anio_mes}_BDX_Marine.xlsx tienen más de una fila. '
              f'Se suman todas: {n_netean_cero} netean a ~0 (no generan diferencia en Siniestro), '
              f'{n_no_netean} NO netean a 0 (reserva real repartida en varias filas — revisar si aplica).')

    return (
        df_bdx_claim.groupby(['INWARD POLICY N°', 'CLAIM NUMBER'], as_index=False)['Net Reserve (USD)']
        .sum()
        .rename(columns={
            'INWARD POLICY N°': 'Policy Number',
            'CLAIM NUMBER': 'Claim Number',
            'Net Reserve (USD)': 'BDX Reserve',
        })
    )


def _resumen_zereos_por_poliza_validacion(df_log):
    '''
    Agrega el Log_Zereo_OSLR (a nivel CLAIM NUMBER) a nivel póliza, para
    poder pegarlo junto a la comparación Reserve Database vs BDX.

    Parámetros
    ----------
    df_log : pandas.DataFrame
        df_log_zereo_oslr, ya en memoria (Sección 6). Ya incluye INWARD
        POLICY N° porque cols_detalle de _zerear_con_log la agrega cuando
        existe en df_update_db.

    Retorna
    -------
    pandas.DataFrame
        Columnas: INWARD POLICY N°, Claims Zereados, Monto Zereado, Reglas
        Aplicadas. Vacío (mismas columnas) si no hubo zereos o si el log no
        trae la columna de póliza.
    '''
    cols_salida = ['INWARD POLICY N°', 'Claims Zereados', 'Monto Zereado', 'Reglas Aplicadas']
    if df_log.empty or 'INWARD POLICY N°' not in df_log.columns:
        return pd.DataFrame(columns=cols_salida)

    df_log_poliza = df_log.copy()
    df_log_poliza['INWARD POLICY N°'] = df_log_poliza['INWARD POLICY N°'].astype(str).str.strip()

    return (
        df_log_poliza.groupby('INWARD POLICY N°')
        .agg(**{
            'Claims Zereados': ('CLAIM NUMBER', 'nunique'),
            'Monto Zereado': ('OSLR ANTES', 'sum'),
            'Reglas Aplicadas': ('REGLA', lambda s: ', '.join(sorted(s.unique()))),
        })
        .reset_index()
    )


def _construir_validacion(df_reserve, df_bdx, df_log):
    '''
    Cruza Reserve Database (OSLR ya calculado por este notebook) contra BDX
    (Net Reserve reportado por el cedente), a nivel póliza, y le pega el
    contexto de zereos (cuántos claims / cuánto dinero / qué reglas) más el
    cálculo crudo sin ajustes (OSLR CALC).

    Parámetros
    ----------
    df_reserve : pandas.DataFrame
        df_update_db ya en memoria, con INWARD POLICY N° y OSLR Inward en su
        forma final (después del rename de la Sección 8).
    df_bdx : pandas.DataFrame
        Salida de _cargar_bdx_net_reserve_validacion.
    df_log : pandas.DataFrame
        df_log_zereo_oslr, ya en memoria.

    Retorna
    -------
    tuple de pandas.DataFrame
        (validacion, bdx_detalle, resumen_zereos). validacion trae las
        columnas en este orden: Policy Number, BDX, OSLR CALC, Reserve
        Database, Diferencia sin Ajustes, Diferencia, Claims Zereados,
        Monto Zereado, Reglas Aplicadas.
    '''
    reserve_db = (
        df_reserve.groupby('INWARD POLICY N°', as_index=False)['OSLR Inward']
        .sum()
        .rename(columns={'OSLR Inward': 'Reserve Database'})
    )
    resumen_zereos = _resumen_zereos_por_poliza_validacion(df_log)

    bdx_detalle = df_bdx.groupby(['Policy Number', 'Cedant', 'Cover'], as_index=False)['Net Reserve'].sum()
    bdx_total = (
        df_bdx.groupby('Policy Number', as_index=False)['Net Reserve']
        .sum()
        .rename(columns={'Net Reserve': 'BDX'})
    )

    validacion = bdx_total.merge(reserve_db, left_on='Policy Number', right_on='INWARD POLICY N°', how='outer')
    validacion['Policy Number'] = validacion['Policy Number'].fillna(validacion['INWARD POLICY N°'])
    validacion = validacion.drop(columns=['INWARD POLICY N°'])

    validacion['BDX'] = validacion['BDX'].fillna(0)
    validacion['Reserve Database'] = validacion['Reserve Database'].fillna(0)
    validacion['Diferencia'] = validacion['BDX'] - validacion['Reserve Database']

    validacion = validacion.merge(resumen_zereos, left_on='Policy Number', right_on='INWARD POLICY N°', how='left')
    validacion = validacion.drop(columns=['INWARD POLICY N°'], errors='ignore')
    validacion['Claims Zereados'] = validacion['Claims Zereados'].fillna(0).astype(int)
    validacion['Monto Zereado'] = validacion['Monto Zereado'].fillna(0)
    validacion['Reglas Aplicadas'] = validacion['Reglas Aplicadas'].fillna('')

    # Reserva cruda, sin nuestros ajustes de zereo. Supone que cada regla
    # anula el claim por completo (OSLR Inward -> 0), no lo reduce parcial.
    validacion['OSLR CALC'] = validacion['Reserve Database'] + validacion['Monto Zereado']
    validacion['Diferencia sin Ajustes'] = validacion['BDX'] - validacion['OSLR CALC']

    validacion = validacion.sort_values(by='Diferencia', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

    # Orden final de columnas.
    validacion = validacion[[
        'Policy Number', 'BDX', 'OSLR CALC', 'Reserve Database',
        'Diferencia sin Ajustes', 'Diferencia',
        'Claims Zereados', 'Monto Zereado', 'Reglas Aplicadas',
    ]]

    return validacion, bdx_detalle, resumen_zereos


def _cargar_outstanding_payments_validacion(route_account, anio_mes):
    '''
    Carga Payments_{anio_mes}.xlsx (el mismo archivo contable que se lee en
    la Sección 3) y devuelve el monto Outstanding por Claim Number, filtrando
    solo las filas en estado "Outstanding" (se excluyen "Paid" y cualquier
    otro estado).

    El archivo trae varias hojas (AE = Cumulative Expenses Paid, CL =
    Cumulative Claims Paid, VA = Cumulative Valuation Expenses). Esta hoja
    de validación (Outstanding Payments) solo debe reflejar reserva de
    SINIESTROS pendiente de pago, así que se usa EXCLUSIVAMENTE la hoja
    'CL' — AE y VA quedan fuera a propósito (gastos y valuación no son
    Outstanding de siniestro).

    También devuelve el Policy Number reportado en Payments (columna
    PAYMENTS_COL_POLICY) para cada claim. Esto es necesario porque hay
    claims Outstanding que no existen en Reserve Database (df_update_db) —
    sin este dato no habría forma de saber a qué póliza pertenecen y se
    perderían de la hoja Outstanding Payments (ver _construir_detalle_claims).

    Parámetros
    ----------
    route_account : str
        Carpeta de contabilidad del periodo (misma variable de la Sección 3).
    anio_mes : int
        Periodo a validar, formato AAAAMM.

    Retorna
    -------
    pandas.DataFrame
        Columnas: Claim Number, Policy Number (Payments), Outstanding. Claim
        Number y Policy Number (Payments) normalizados (str sin espacios),
        Outstanding numérico sin NaN, agregado (sum) por si un mismo claim
        tiene varias filas Outstanding (en la misma hoja o en hojas
        distintas). Si un mismo Claim Number trae más de un Policy Number
        distinto entre sus filas, se avisa por consola y se usa el primero
        que aparece (dato de Payments inconsistente, hay que revisarlo).

    Excepciones
    -----------
    FileNotFoundError
        Si no existe Payments_{anio_mes}.xlsx.
    KeyError
        Si el archivo existe pero no trae la hoja 'CL', o si la hoja 'CL'
        no trae PAYMENTS_COL_CLAIM, PAYMENTS_COL_STATUS, PAYMENTS_COL_AMOUNT
        o PAYMENTS_COL_POLICY. Ajusta esos 4 nombres al principio de la
        sección si no calzan con las columnas reales.
    '''
    path = f'{route_account}/Payments_{anio_mes}.xlsx'
    if not os.path.exists(path):
        raise FileNotFoundError(f'No se encontró Payments_{anio_mes}.xlsx en {path}.')

    hojas = pd.read_excel(path, sheet_name=None)
    if 'CL' not in hojas:
        raise KeyError(
            f"Payments_{anio_mes}.xlsx no trae la hoja 'CL' (Cumulative Claims Paid). "
            f"Hojas encontradas: {list(hojas.keys())}."
        )
    df_payments = hojas['CL'].copy()
    print(f"Payments_{anio_mes}.xlsx: usando solo la hoja 'CL' (Cumulative Claims Paid) "
          f"para Outstanding Payments — {len(df_payments)} filas.")

    faltantes = {PAYMENTS_COL_CLAIM, PAYMENTS_COL_STATUS, PAYMENTS_COL_AMOUNT, PAYMENTS_COL_POLICY} - set(df_payments.columns)
    if faltantes:
        raise KeyError(
            f"Payments_{anio_mes}.xlsx (hoja 'CL') no trae las columnas esperadas: {faltantes}. "
            'Ajusta PAYMENTS_COL_CLAIM / PAYMENTS_COL_STATUS / PAYMENTS_COL_AMOUNT / PAYMENTS_COL_POLICY al inicio de la sección.'
        )

    df_outstanding = df_payments[
        df_payments[PAYMENTS_COL_STATUS].astype(str).str.strip().str.casefold() == 'outstanding'
    ].copy()

    df_outstanding[PAYMENTS_COL_AMOUNT] = pd.to_numeric(df_outstanding[PAYMENTS_COL_AMOUNT], errors='coerce').fillna(0)
    df_outstanding[PAYMENTS_COL_CLAIM] = _limpiar_llave(df_outstanding[PAYMENTS_COL_CLAIM])
    df_outstanding[PAYMENTS_COL_POLICY] = _limpiar_llave(df_outstanding[PAYMENTS_COL_POLICY])

    n_policies_por_claim = df_outstanding.groupby(PAYMENTS_COL_CLAIM)[PAYMENTS_COL_POLICY].nunique()
    claims_inconsistentes = n_policies_por_claim[n_policies_por_claim > 1]
    if not claims_inconsistentes.empty:
        print(f'Aviso: {len(claims_inconsistentes)} Claim Number en Payments_{anio_mes}.xlsx traen más de un '
              f'Policy Number distinto entre sus filas ({", ".join(claims_inconsistentes.index[:10])}'
              f'{"..." if len(claims_inconsistentes) > 10 else ""}). Se usa el primero que aparece por claim.')

    resultado = df_outstanding.groupby(PAYMENTS_COL_CLAIM, as_index=False).agg(**{
        PAYMENTS_COL_AMOUNT: (PAYMENTS_COL_AMOUNT, 'sum'),
        PAYMENTS_COL_POLICY: (PAYMENTS_COL_POLICY, 'first'),
    })

    return resultado.rename(columns={
        PAYMENTS_COL_CLAIM: 'Claim Number',
        PAYMENTS_COL_AMOUNT: 'Outstanding',
        PAYMENTS_COL_POLICY: 'Policy Number (Payments)',
    })


def _construir_detalle_claims(df_reserve, df_bdx_claim, df_outstanding):
    '''
    Arma el detalle base por Claim Number (Policy Number, Claim Number,
    Reserve Database, BDX Reserve, Outstanding, Diferencia por Tipo de
    Cambio), SIN filtrar filas. Es la base compartida de la que se derivan
    tanto la hoja Siniestro (filtrada por diferencia) como la hoja
    Outstanding Payments (filtrada por Outstanding > 0).

    Cruza TRES fuentes por Claim Number, todas con merge OUTER (no left),
    porque cada una puede traer claims que las otras no conocen: Reserve
    Database (df_reserve/df_update_db) no tiene los claims que aún no se
    cargaron a la base; el BDX real (df_bdx_claim/{AñoMes}_BDX_Marine.xlsx)
    puede traer claims que la base todavía no tiene; Payments (Outstanding)
    puede traer claims de otra cartera. Con un merge left (como en la
    versión anterior) esos claims se perdían en silencio. Reserve Database,
    BDX Reserve y Outstanding quedan en 0 cuando su fuente no trae el claim.

    BDX Reserve es el Net Reserve (USD) REAL por claim de {AñoMes}_BDX_Marine.xlsx
    — YA NO se reparte proporcionalmente el total de la póliza (como se hacía
    antes con el único BDX disponible, OSLR {AñoMes}.xlsx, que solo viene a
    nivel póliza). Ese archivo sigue usándose, sin cambios, en la pestaña
    Validación.

    Resolución de Policy Number: prioriza Reserve Database (dato interno más
    confiable); si el claim no está ahí, usa el Policy Number de
    {AñoMes}_BDX_Marine.xlsx; si tampoco está ahí, usa el de Payments; si no hay
    Policy Number en ninguna fuente, se usa un placeholder
    "SIN POLIZA - <Claim Number>" y se avisa por consola para investigar por
    qué el claim no está en la base.

    Diferencia por Tipo de Cambio = Reserve Database - BDX Reserve (con
    signo, YA NO es MAX(3 valores) ni involucra Outstanding). Este cambio
    hace que sumar las diferencias de todos los claims de una póliza
    reproduzca el mismo monto que compararías manualmente contra esa
    póliza en Detalle BDX (Net Reserve) vs Reserve Database — la fórmula
    anterior (MAX de 3 valores) no era aditiva, por eso "explotaba" con
    diferencias más grandes que las de Validación aun cuando el problema
    real era simple. Verificado exacto contra 9 claims reales de la
    póliza 3612100000008 (ej. claim 324372640000403: Reserve Database
    404.34 - BDX Reserve 28154.95217 = -27750.61217, que es justo la
    diferencia que se buscaba). Outstanding se sigue calculando en este
    detalle base (lo usa la hoja Outstanding Payments) pero ya NO
    participa en esta resta.

    Parámetros
    ----------
    df_reserve : pandas.DataFrame
        df_update_db ya en memoria (INWARD POLICY N°, CLAIM NUMBER, OSLR
        Inward).
    df_bdx_claim : pandas.DataFrame
        Salida de _cargar_bdx_reserve_por_claim_validacion (Policy Number,
        Claim Number, BDX Reserve).
    df_outstanding : pandas.DataFrame
        Salida de _cargar_outstanding_payments_validacion (Claim Number,
        Policy Number (Payments), Outstanding).

    Retorna
    -------
    pandas.DataFrame
        Columnas, en este orden: Policy Number, Claim Number, Reserve
        Database, BDX Reserve, Outstanding, Diferencia por Tipo de Cambio,
        En Reserve Database (bool, auxiliar — True si esa póliza existe en
        Reserve Database; la usa _construir_siniestro_detalle y se descarta
        antes de exportar). Incluye la UNIÓN de claims de las tres fuentes,
        sin filtrar.
    '''
    reserve_claim = (
        df_reserve.groupby(['INWARD POLICY N°', 'CLAIM NUMBER'], as_index=False)['OSLR Inward']
        .sum()
        .rename(columns={
            'INWARD POLICY N°': 'Policy Number',
            'CLAIM NUMBER': 'Claim Number',
            'OSLR Inward': 'Reserve Database',
        })
    )
    reserve_claim['Policy Number'] = _limpiar_llave(reserve_claim['Policy Number']).replace(POLICY_NUMBER_RENAME)
    reserve_claim['Claim Number'] = _limpiar_llave(reserve_claim['Claim Number'])

    # Set de pólizas que SÍ están en Reserve Database (post rename), para el
    # filtro de la hoja Siniestro (_construir_siniestro_detalle): solo debe
    # mostrar claims de pólizas "con las que estamos trabajando", o sea que
    # tengan Reserve Database, no las que solo vienen del BDX real o de
    # Payments.
    polizas_en_reserve_db = set(reserve_claim['Policy Number'].unique())

    df_bdx_claim = df_bdx_claim.copy()
    df_bdx_claim['Policy Number'] = _limpiar_llave(df_bdx_claim['Policy Number']).replace(POLICY_NUMBER_RENAME)
    df_outstanding = df_outstanding.copy()
    df_outstanding['Policy Number (Payments)'] = _limpiar_llave(df_outstanding['Policy Number (Payments)']).replace(POLICY_NUMBER_RENAME)

    # Merge OUTER con el BDX real por claim: conserva los claims que solo
    # están en {AñoMes}_BDX_Marine.xlsx aunque Reserve Database no los tenga.
    detalle = reserve_claim.merge(
        df_bdx_claim, on='Claim Number', how='outer', suffixes=('', ' (BDX)')
    )
    detalle['Reserve Database'] = detalle['Reserve Database'].fillna(0)
    detalle['BDX Reserve'] = detalle['BDX Reserve'].fillna(0)
    detalle['Policy Number'] = detalle['Policy Number'].fillna(detalle['Policy Number (BDX)'])
    detalle = detalle.drop(columns=['Policy Number (BDX)'])

    # Merge OUTER con Outstanding: conserva los claims que solo están en
    # Payments (Outstanding) aunque las dos fuentes anteriores no los tengan.
    detalle = detalle.merge(df_outstanding, on='Claim Number', how='outer')
    detalle['Reserve Database'] = detalle['Reserve Database'].fillna(0)
    detalle['BDX Reserve'] = detalle['BDX Reserve'].fillna(0)
    detalle['Outstanding'] = detalle['Outstanding'].fillna(0)

    sin_policy = detalle['Policy Number'].isna()
    if sin_policy.any():
        n_huerfanos = sin_policy.sum()
        detalle['Policy Number'] = detalle['Policy Number'].fillna(detalle['Policy Number (Payments)'])
        print(f'Aviso: {n_huerfanos} Claim Number no están en Reserve Database ni en el BDX real por '
              'claim (archivo BDX Marine); se agregan igual usando su Policy Number de Payments.')

    sin_policy_definitivo = detalle['Policy Number'].isna() | (detalle['Policy Number'].astype(str).str.strip() == '')
    if sin_policy_definitivo.any():
        detalle.loc[sin_policy_definitivo, 'Policy Number'] = (
            'SIN POLIZA - ' + detalle.loc[sin_policy_definitivo, 'Claim Number'].astype(str)
        )
        print(f'Aviso: {sin_policy_definitivo.sum()} Claim Number quedaron sin Policy Number en ninguna '
              f'de las tres fuentes. Se marcan como "SIN POLIZA - <Claim Number>" — revisar el dato origen.')

    detalle = detalle.drop(columns=['Policy Number (Payments)'])
    detalle['Policy Number'] = _limpiar_llave(detalle['Policy Number']).replace(POLICY_NUMBER_RENAME)

    # Reserve Database - BDX Reserve, con signo. Outstanding NO participa
    # (ver docstring del cambio de fórmula).
    detalle['Diferencia por Tipo de Cambio'] = detalle['Reserve Database'] - detalle['BDX Reserve']

    # Columna auxiliar (no se exporta tal cual): True si la póliza del claim
    # existe en Reserve Database. La usa _construir_siniestro_detalle para
    # quedarse solo con las pólizas "con las que estamos trabajando".
    detalle['En Reserve Database'] = detalle['Policy Number'].isin(polizas_en_reserve_db)

    return detalle[[
        'Policy Number', 'Claim Number', 'Reserve Database', 'BDX Reserve',
        'Outstanding', 'Diferencia por Tipo de Cambio', 'En Reserve Database',
    ]]


def _construir_siniestro_detalle(detalle_claims, tolerancia=1.0):
    '''
    Filtra el detalle base (_construir_detalle_claims) a los claims de
    pólizas "con las que estamos trabajando" (columna auxiliar 'En Reserve
    Database': existen en Reserve Database/df_update_db, se descartan
    claims cuya póliza únicamente viene del BDX real o de Payments), y
    calcula 3 columnas nuevas a partir de Reserve Database, BDX Reserve y
    Outstanding:
      - Diferencias = |Reserve Database - BDX Reserve| (valor absoluto).
      - Outstanding Payments = el monto Outstanding de ese mismo claim (ya
        cruzado en _construir_detalle_claims); se renombra solo para que el
        nombre de columna sea claro en el reporte.
      - Diferencia por Tipo de Cambio = Outstanding Payments - Diferencias
        (columna informativa; YA NO se usa para decidir inclusión — ver
        siguiente párrafo).

    Criterio de inclusión (2026-07-20, corregido): solo Diferencias >
    tolerancia. Antes el criterio exigía ADEMÁS |Diferencia por Tipo de
    Cambio| > tolerancia, lo que descartaba claims con una discrepancia
    real grande entre Reserve Database y BDX Reserve cuando el Outstanding
    Payment del mismo claim casi la compensaba en esa resta (ej. Diferencias
    22,043.98 con Outstanding Payments ~22,043 => Diferencia por Tipo de
    Cambio ~0, quedaba fuera aunque la discrepancia real era grande). Esto
    hacía que el total de la hoja Siniestro no reconciliara contra la
    pestaña Validación (caso real: 3 claims de la póliza E01-2-60-10 por
    $61,821.49 desaparecían de la hoja aun siendo parte de los $334,030.47
    de diferencia validados para esa póliza). Con el criterio único
    (Diferencias > tolerancia), todo claim con una discrepancia real entre
    Reserve Database y BDX Reserve aparece, y la suma de Diferencias por
    póliza vuelve a reconciliar con Validación.

    Parámetros
    ----------
    detalle_claims : pandas.DataFrame
        Salida de _construir_detalle_claims (incluye las columnas
        auxiliares 'Outstanding' y 'En Reserve Database').
    tolerancia : float, default 1.0
        Diferencia máxima (USD) aceptable antes de incluir el claim.
        Mismo criterio que la pestaña Validación.

    Retorna
    -------
    pandas.DataFrame
        Columnas, en este orden: Policy Number, Claim Number, Reserve
        Database, BDX Reserve, Diferencias, Outstanding Payments,
        Diferencia por Tipo de Cambio. Solo pólizas en Reserve Database y
        con Diferencias > tolerancia, ordenado por Diferencias (mayor a
        menor).
    '''
    detalle = detalle_claims[detalle_claims['En Reserve Database']].copy()

    detalle['Diferencias'] = (detalle['Reserve Database'] - detalle['BDX Reserve']).abs()
    detalle = detalle.rename(columns={'Outstanding': 'Outstanding Payments'})
    detalle['Diferencia por Tipo de Cambio'] = detalle['Outstanding Payments'] - detalle['Diferencias']

    # Único criterio de inclusión: que haya una discrepancia real entre
    # Reserve Database y BDX Reserve que supere la tolerancia. Ya no se
    # exige además que 'Diferencia por Tipo de Cambio' supere la tolerancia
    # (ver docstring: ese criterio ocultaba diferencias reales grandes
    # cuando el Outstanding Payment las compensaba en la resta).
    detalle = detalle[detalle['Diferencias'] > tolerancia].copy()
    detalle = detalle.sort_values(
        by='Diferencias', ascending=False
    ).reset_index(drop=True)

    return detalle[[
        'Policy Number', 'Claim Number', 'Reserve Database', 'BDX Reserve',
        'Diferencias', 'Outstanding Payments', 'Diferencia por Tipo de Cambio',
    ]]


def _construir_outstanding_payments_detalle(detalle_claims):
    '''
    Filtra el detalle base (_construir_detalle_claims) a los claims que
    tienen un pago Outstanding (Outstanding > 0) Y cuya póliza existe en
    Reserve Database (columna auxiliar 'En Reserve Database' — mismo
    criterio marine que usa la hoja Siniestro), para la hoja Outstanding
    Payments. Antes esta hoja no aplicaba el filtro de Reserve Database y
    mostraba cualquier claim con Outstanding aunque fuera de otra línea de
    negocio; ahora queda acotada a pólizas marine igual que Siniestro.
    Sigue sin incluir la columna 'Diferencia por Tipo de Cambio' — esta hoja
    es solo para ver qué hay pendiente de pago, no para la comparación BDX
    vs Reserve Database.

    Parámetros
    ----------
    detalle_claims : pandas.DataFrame
        Salida de _construir_detalle_claims.

    Retorna
    -------
    pandas.DataFrame
        Columnas: Policy Number, Claim Number, Reserve Database, BDX
        Reserve, Outstanding (sin 'Diferencia por Tipo de Cambio' ni 'En
        Reserve Database'). Solo pólizas en Reserve Database (marine) con
        Outstanding > 0, ordenado por Outstanding (mayor a menor).
    '''
    detalle = detalle_claims[
        (detalle_claims['Outstanding'] > 0) & detalle_claims['En Reserve Database']
    ].copy()
    detalle = detalle.drop(columns=['Diferencia por Tipo de Cambio', 'En Reserve Database'])
    return detalle.sort_values(by='Outstanding', ascending=False).reset_index(drop=True)


def _construir_resumen_validacion(validacion, polizas_marine, detalle_claims=None):
    '''
    Arma una vista resumen por Policy Number, tipo tabla dinámica, para la
    hoja Resumen: una fila por póliza con el total de BDX, Reserve Database
    y Diferencia (misma fuente que la hoja Validación), más el total de
    Outstanding Payments y de Diferencia por Tipo de Cambio agregados desde
    el detalle de claims. Cierra con una fila 'Total general' que suma cada
    columna, para poder cuadrar rápido contra el total de Validación sin
    abrir esa hoja.

    A diferencia de la hoja Validación (que NO se filtra, para conservar
    visibilidad completa del BDX), Resumen SOLO incluye pólizas marine
    (Policy Number en polizas_marine): pólizas de otras líneas de negocio
    que aparecen en Validación (porque están en el BDX pero no en Reserve
    Database) quedan fuera de Resumen a propósito. El detalle de claims se
    filtra igual, usando su propia columna 'En Reserve Database' (mismo
    criterio marine que las hojas Siniestro/Outstanding Payments).

    'Diferencia por Tipo de Cambio' usa la MISMA fórmula que la hoja
    Siniestro (_construir_siniestro_detalle), calculada a nivel claim y
    luego sumada por póliza: Diferencias = |Reserve Database - BDX Reserve|,
    Diferencia por Tipo de Cambio = Outstanding Payments - Diferencias.
    Antes Resumen usaba la fórmula del detalle base (_construir_detalle_claims:
    Reserve Database - BDX Reserve, con signo, sin Outstanding), que no es la
    misma que se ve en Siniestro y no se podía reconciliar contra esa hoja.
    A diferencia de Siniestro, aquí NO se filtra por tolerancia — se suman
    TODOS los claims marine de la póliza, no solo los que exceden tolerancia.

    Cruce Resumen (Validación) + detalle de claims: 'validacion' trae Policy
    Number normalizado solo con .strip() (conserva espacios internos, ej.
    "90600 320575"), mientras que detalle_claims (via _construir_detalle_claims)
    usa _limpiar_llave + POLICY_NUMBER_RENAME (sin NINGÚN espacio, ej.
    "90600320575"). Cruzar directo por 'Policy Number' con esos dos formatos
    duplicaba la póliza en Resumen: una fila con espacios (BDX/Reserve
    Database) y otra sin espacios (Outstanding/Diferencia por Tipo de
    Cambio). Se cruza por una llave normalizada (_policy_key, aplicando el
    mismo _limpiar_llave + POLICY_NUMBER_RENAME a ambos lados) y se conserva
    como 'Policy Number' visible el formato de 'validacion' (el que se usa
    en el resto del reporte), cayendo al de detalle_claims solo si la
    póliza no está en 'validacion'.

    Parámetros
    ----------
    validacion : pandas.DataFrame
        Salida de _construir_validacion. Debe traer 'Policy Number', 'BDX',
        'Reserve Database' y 'Diferencia'.
    polizas_marine : set
        Policy Number (mismo formato que 'validacion': str + strip, sin
        des-espaciar) de las pólizas consideradas marine — típicamente
        df_update_db['INWARD POLICY N°'] ya normalizado. Filtra Resumen a
        estas pólizas.
    detalle_claims : pandas.DataFrame o None, default None
        Salida de _construir_detalle_claims (detalle base por Claim Number,
        con 'Policy Number', 'Outstanding', 'Reserve Database', 'BDX
        Reserve' y 'En Reserve Database'). Se filtra internamente a 'En
        Reserve Database' antes de agregar. Si es None o viene vacío (ej.
        no hubo Payments o BDX por claim para el periodo), 'Outstanding
        Payments' y 'Diferencia por Tipo de Cambio' salen en 0 para todas
        las pólizas — el resumen no se detiene por esto.

    Retorna
    -------
    pandas.DataFrame
        Columnas, en este orden: Policy Number, Suma de BDX, Suma de
        Reserve Database, Suma de Diferencia, Outstanding Payments,
        Diferencia por Tipo de Cambio. Una fila por póliza marine (orden
        alfabético, sin duplicados por formato de Policy Number) más una
        fila final 'Total general' con la suma de cada columna numérica.
    '''
    cols_num = [
        'Suma de BDX', 'Suma de Reserve Database', 'Suma de Diferencia',
        'Outstanding Payments', 'Diferencia por Tipo de Cambio',
    ]
    if detalle_claims is not None and not detalle_claims.empty:
        detalle_marine = detalle_claims[detalle_claims['En Reserve Database']].copy()
        # Mismo criterio que Siniestro, a nivel claim (ver docstring).
        detalle_marine['Diferencias'] = (detalle_marine['Reserve Database'] - detalle_marine['BDX Reserve']).abs()
        detalle_marine['Diferencia por Tipo de Cambio'] = detalle_marine['Outstanding'] - detalle_marine['Diferencias']
        agregado_claims = detalle_marine.groupby('Policy Number', as_index=False).agg(**{
            'Outstanding Payments': ('Outstanding', 'sum'),
            'Diferencia por Tipo de Cambio': ('Diferencia por Tipo de Cambio', 'sum'),
        })
    else:
        agregado_claims = pd.DataFrame(columns=['Policy Number', 'Outstanding Payments', 'Diferencia por Tipo de Cambio'])

    # Llave normalizada para el cruce (ver docstring): evita duplicar la
    # póliza cuando 'validacion' y detalle_claims traen formatos distintos
    # de Policy Number para la misma póliza real.
    validacion_marine = validacion[validacion['Policy Number'].isin(polizas_marine)].copy()
    validacion_marine['_policy_key'] = _limpiar_llave(validacion_marine['Policy Number']).replace(POLICY_NUMBER_RENAME)
    agregado_claims['_policy_key'] = _limpiar_llave(agregado_claims['Policy Number']).replace(POLICY_NUMBER_RENAME)

    resumen = validacion_marine[['_policy_key', 'Policy Number', 'BDX', 'Reserve Database', 'Diferencia']].rename(columns={
        'BDX': 'Suma de BDX',
        'Reserve Database': 'Suma de Reserve Database',
        'Diferencia': 'Suma de Diferencia',
    })
    resumen = resumen.merge(
        agregado_claims[['_policy_key', 'Policy Number', 'Outstanding Payments', 'Diferencia por Tipo de Cambio']],
        on='_policy_key', how='outer', suffixes=('', ' (claims)'),
    )
    resumen['Policy Number'] = resumen['Policy Number'].fillna(resumen['Policy Number (claims)'])
    resumen = resumen.drop(columns=['_policy_key', 'Policy Number (claims)'])
    resumen[cols_num] = resumen[cols_num].fillna(0)
    resumen = resumen.sort_values('Policy Number').reset_index(drop=True)

    total_general = {'Policy Number': 'Total general', **{col: resumen[col].sum() for col in cols_num}}
    resumen = pd.concat([resumen, pd.DataFrame([total_general])], ignore_index=True)
    return resumen[['Policy Number'] + cols_num]


def _exportar_validacion(validacion, bdx_detalle, df_log, resumen_zereos, siniestro_detalle, outstanding_detalle,
                          resumen, polizas_marine, ruta_salida, anio_mes, tolerancia=1.0,
                          df_reconciliacion_AE=None, df_reconciliacion_CL=None, df_reconciliacion_VA=None):
    '''
    Exporta el resultado a Excel con 8 hojas, en este orden de pestaña:
    Validación, Detalle BDX, Log Zereo OSLR (oculta), Zereos por Póliza
    (oculta), Siniestro, Outstanding Payments, Payments, Resumen (esta
    última se crea al final a propósito, para que quede como última pestaña
    visible del archivo). Resalta en rojo las pólizas con diferencia sin explicar y
    en amarillo las que ya tienen un zereo que la explica. 'Log Zereo OSLR' y
    'Zereos por Póliza' se generan OCULTAS: el detalle sigue disponible
    dentro del archivo, pero no estorba al abrirlo (se puede mostrar de
    nuevo con clic derecho sobre cualquier pestaña visible > Mostrar).

    Parámetros
    ----------
    validacion, bdx_detalle, df_log, resumen_zereos : pandas.DataFrame
        Salidas de _construir_validacion (y el log crudo).
    siniestro_detalle : pandas.DataFrame
        Salida de _construir_siniestro_detalle (claims con diferencia por
        tipo de cambio fuera de tolerancia). Puede venir vacía (mismas
        columnas) si no hubo Payments para el periodo.
    outstanding_detalle : pandas.DataFrame
        Salida de _construir_outstanding_payments_detalle (todos los claims
        con Outstanding > 0, sin filtrar por diferencia). Mismas columnas
        que siniestro_detalle. Puede venir vacía si no hubo Payments.
    resumen : pandas.DataFrame
        Salida de _construir_resumen_validacion (una fila por póliza con
        Suma de BDX / Reserve Database / Diferencia / Outstanding Payments /
        Diferencia por Tipo de Cambio, más la fila 'Total general'). Se
        exporta como la primera hoja del archivo.
    ruta_salida : str
        Ruta completa del .xlsx a generar.
    anio_mes : int
        Periodo validado (para el encabezado del reporte).
    tolerancia : float, default 1.0
        Diferencia máxima (USD) aceptable antes de marcar la póliza.
    polizas_marine : set
        Policy Number (mismo formato que 'validacion') de las pólizas
        marine. Se usa SOLO para el chequeo de coherencia entre 'Resumen'
        (ya filtrado a marine) y 'Validación' (se exporta completa, sin
        filtrar) — no cambia qué filas se exportan a Excel.
    df_reconciliacion_AE, df_reconciliacion_CL, df_reconciliacion_VA : pandas.DataFrame, opcional
        Salida de _reconciliar_payments_subsidiaria (Sección 4) para cada
        subsidiaria. Columnas 'Policy N°', 'dfcontafinal', 'PAYMENTS AñoMes'.
        Si se omite alguna (None), esa subsidiaria se exporta vacía en la
        hoja Payments en vez de fallar.

    Retorna
    -------
    None

    Excepciones
    -----------
    OSError
        Si no se puede escribir el archivo (ej. está abierto en Excel).
    '''
    cols_reconciliacion_vacio = ['Policy N°', 'dfcontafinal', 'PAYMENTS AñoMes']
    df_reconciliacion_AE = df_reconciliacion_AE if df_reconciliacion_AE is not None else pd.DataFrame(columns=cols_reconciliacion_vacio)
    df_reconciliacion_CL = df_reconciliacion_CL if df_reconciliacion_CL is not None else pd.DataFrame(columns=cols_reconciliacion_vacio)
    df_reconciliacion_VA = df_reconciliacion_VA if df_reconciliacion_VA is not None else pd.DataFrame(columns=cols_reconciliacion_vacio)

    def _construir_payments_wide(df_ae, df_cl, df_va):
        '''
        Combina las 3 reconciliaciones por subsidiaria (AE, CL, VA) en una
        sola tabla ancha indexada por Claims Reference (numero de
        siniestro), para la hoja Payments.

        Parametros
        ----------
        df_ae, df_cl, df_va : pandas.DataFrame
            Columnas 'Claims Reference', 'dfcontafinal', 'PAYMENTS AñoMes'
            (pueden venir vacias, mismas columnas).

        Retorna
        -------
        pandas.DataFrame
            Columnas MultiIndex (subsidiaria, 'dfcontafinal'/'PAYMENTS
            AñoMes'). Index: Claims Reference (union ordenada de las 3
            fuentes). Celdas sin dato en alguna subsidiaria quedan en 0.
        '''
        partes = []
        for nombre, df in [('AE', df_ae), ('CL', df_cl), ('VA', df_va)]:
            d = df.set_index('Claims Reference')[['dfcontafinal', 'PAYMENTS AñoMes']].copy()
            d.columns = pd.MultiIndex.from_product([[nombre], d.columns])
            partes.append(d)
        payments_wide = pd.concat(partes, axis=1).fillna(0).sort_index()
        payments_wide.index.name = 'Claim Number'
        return payments_wide

    try:
        with pd.ExcelWriter(ruta_salida, engine='xlsxwriter') as writer:
            # 'Resumen' se crea al final (ver mas abajo, despues de la hoja
            # Payments) para que quede como ultima pestaña visible del archivo.
            validacion.to_excel(writer, sheet_name='Validación', index=False, startrow=1)
            bdx_detalle.to_excel(writer, sheet_name='Detalle BDX', index=False)
            df_log.to_excel(writer, sheet_name='Log Zereo OSLR', index=False)
            resumen_zereos.to_excel(writer, sheet_name='Zereos por Póliza', index=False)
            siniestro_detalle.to_excel(writer, sheet_name='Siniestro', index=False, startrow=1)
            outstanding_detalle.to_excel(writer, sheet_name='Outstanding Payments', index=False, startrow=1)

            workbook = writer.book

            fmt_header = workbook.add_format({'bold': True, 'bg_color': '#D9E1F2', 'border': 1})
            fmt_money = workbook.add_format({'num_format': '$#,##0.00'})
            # fmt_alerta / fmt_explicada se usan para SOBRESCRIBIR celdas de
            # columnas de diferencia (Validación 'Diferencia', Siniestro
            # 'Diferencia por Tipo de Cambio') resaltando su fondo. Si no
            # llevan también 'num_format', la sobreescritura pisa el formato
            # de moneda de la columna y el número sale con todos sus
            # decimales de float en vez de 2 — por eso ambos formatos
            # incluyen el mismo num_format que fmt_money.
            fmt_alerta = workbook.add_format({'bg_color': '#F8CBAD', 'num_format': '$#,##0.00'})
            fmt_explicada = workbook.add_format({'bg_color': '#FFF2CC', 'num_format': '$#,##0.00'})
            fmt_titulo = workbook.add_format({'bold': True, 'font_size': 12})
            fmt_total = workbook.add_format({'bold': True, 'num_format': '$#,##0.00', 'top': 2})
            fmt_total_label = workbook.add_format({'bold': True, 'top': 2})

            # --- Hoja Validación ---
            ws = writer.sheets['Validación']
            ws.write(0, 0, f'Validación OSLR vs BDX — Periodo {anio_mes}', fmt_titulo)

            cols_moneda = (
                'BDX', 'OSLR CALC', 'Reserve Database', 'Diferencia sin Ajustes',
                'Diferencia', 'Monto Zereado',
            )
            for col_idx, col_name in enumerate(validacion.columns):
                ws.write(1, col_idx, col_name, fmt_header)
                if col_name in cols_moneda:
                    ws.set_column(col_idx, col_idx, 18, fmt_money)
                else:
                    ws.set_column(col_idx, col_idx, 22)

            diff_col = validacion.columns.get_loc('Diferencia')
            claims_col = validacion.columns.get_loc('Claims Zereados')
            for row_idx, row in enumerate(validacion.itertuples(index=False, name=None), start=2):
                diferencia = row[diff_col]
                claims_zereados = row[claims_col]
                if abs(diferencia) > tolerancia:
                    fmt = fmt_explicada if claims_zereados > 0 else fmt_alerta
                    ws.write(row_idx, diff_col, diferencia, fmt)

            # --- Hoja Siniestro ---
            ws_sin = writer.sheets['Siniestro']
            ws_sin.write(0, 0, f'Siniestros con diferencia por Tipo de Cambio — Periodo {anio_mes}', fmt_titulo)

            cols_moneda_siniestro = (
                'Reserve Database', 'BDX Reserve', 'Outstanding', 'Diferencias',
                'Outstanding Payments', 'Diferencia por Tipo de Cambio',
            )
            for col_idx, col_name in enumerate(siniestro_detalle.columns):
                ws_sin.write(1, col_idx, col_name, fmt_header)
                if col_name in cols_moneda_siniestro:
                    ws_sin.set_column(col_idx, col_idx, 20, fmt_money)
                else:
                    ws_sin.set_column(col_idx, col_idx, 20)

            if not siniestro_detalle.empty:
                diff_tc_col = siniestro_detalle.columns.get_loc('Diferencia por Tipo de Cambio')
                for row_idx, row in enumerate(siniestro_detalle.itertuples(index=False, name=None), start=2):
                    ws_sin.write(row_idx, diff_tc_col, row[diff_tc_col], fmt_alerta)

            # --- Hoja Outstanding Payments ---
            ws_out = writer.sheets['Outstanding Payments']
            ws_out.write(0, 0, f'Claims con Outstanding Payments — Periodo {anio_mes}', fmt_titulo)

            for col_idx, col_name in enumerate(outstanding_detalle.columns):
                ws_out.write(1, col_idx, col_name, fmt_header)
                if col_name in cols_moneda_siniestro:
                    ws_out.set_column(col_idx, col_idx, 20, fmt_money)
                else:
                    ws_out.set_column(col_idx, col_idx, 20)

            # --- Hoja Payments: dfcontafinal vs archivo Payments crudo, por siniestro ---
            payments_wide = _construir_payments_wide(df_reconciliacion_AE, df_reconciliacion_CL, df_reconciliacion_VA)
            ws_pay = workbook.add_worksheet('Payments')
            writer.sheets['Payments'] = ws_pay

            fmt_verde = workbook.add_format({'bg_color': '#C6E0B4', 'num_format': '$#,##0.00'})
            TOLERANCIA_PAYMENTS = 0.01

            ws_pay.merge_range(0, 0, 1, 0, 'Claim Number', fmt_header)
            ws_pay.set_column(0, 0, 22)
            col_idx = 1
            for sub in ('AE', 'CL', 'VA'):
                ws_pay.merge_range(0, col_idx, 0, col_idx + 1, sub, fmt_header)
                ws_pay.write(1, col_idx, 'dfcontafinal', fmt_header)
                ws_pay.write(1, col_idx + 1, 'PAYMENTS AñoMes', fmt_header)
                ws_pay.set_column(col_idx, col_idx + 1, 16)
                col_idx += 2

            for row_idx, (claim_number, fila) in enumerate(payments_wide.iterrows(), start=2):
                ws_pay.write(row_idx, 0, claim_number)
                col_idx = 1
                for sub in ('AE', 'CL', 'VA'):
                    valor_final = fila.get((sub, 'dfcontafinal'), 0)
                    valor_crudo = fila.get((sub, 'PAYMENTS AñoMes'), 0)
                    if abs(valor_final - valor_crudo) > TOLERANCIA_PAYMENTS:
                        ws_pay.write(row_idx, col_idx, valor_final, fmt_verde)
                        ws_pay.write(row_idx, col_idx + 1, valor_crudo, fmt_alerta)
                    else:
                        ws_pay.write(row_idx, col_idx, valor_final, fmt_money)
                        ws_pay.write(row_idx, col_idx + 1, valor_crudo, fmt_money)
                    col_idx += 2

            # --- Hoja Resumen (creada al final: queda como ultima pestaña visible) ---
            resumen.to_excel(writer, sheet_name='Resumen', index=False, startrow=1)
            ws_resumen = writer.sheets['Resumen']
            ws_resumen.write(0, 0, f'Resumen por Póliza — Periodo {anio_mes}', fmt_titulo)

            cols_moneda_resumen = (
                'Suma de BDX', 'Suma de Reserve Database', 'Suma de Diferencia',
                'Outstanding Payments', 'Diferencia por Tipo de Cambio',
            )
            for col_idx, col_name in enumerate(resumen.columns):
                ws_resumen.write(1, col_idx, col_name, fmt_header)
                if col_name in cols_moneda_resumen:
                    ws_resumen.set_column(col_idx, col_idx, 20, fmt_money)
                else:
                    ws_resumen.set_column(col_idx, col_idx, 26)

            # Fila 'Total general' en negrita, con una línea superior para
            # separarla visualmente de las pólizas (la calcula
            # _construir_resumen_validacion, no es una fórmula de Excel).
            fila_total = len(resumen) + 1
            for col_idx, col_name in enumerate(resumen.columns):
                valor = resumen.iloc[-1, col_idx]
                fmt = fmt_total if col_name in cols_moneda_resumen else fmt_total_label
                ws_resumen.write(fila_total, col_idx, valor, fmt)

            # --- Ocultar hojas de detalle (siguen en el archivo, no estorban) ---
            writer.sheets['Log Zereo OSLR'].hide()
            writer.sheets['Zereos por Póliza'].hide()

        n_excede = validacion['Diferencia'].abs() > tolerancia
        n_explicadas = (n_excede & (validacion['Claims Zereados'] > 0)).sum()
        n_sin_explicar = (n_excede & (validacion['Claims Zereados'] == 0)).sum()

        print(f'Validación OSLR vs BDX exportada: {ruta_salida}')
        print(f'  Pólizas dentro de tolerancia (${tolerancia}): {(~n_excede).sum()}')
        print(f'  Con diferencia y CON zereo que podría explicarla: {n_explicadas}')
        print(f'  Con diferencia SIN ningún zereo que la explique (revisar primero): {n_sin_explicar}')
        print(f'  Claims en hoja Siniestro con diferencia por tipo de cambio (>${tolerancia}): {len(siniestro_detalle)}')
        print(f'  Claims en hoja Outstanding Payments (Outstanding > 0): {len(outstanding_detalle)}')

        # Chequeo de coherencia: el total de la hoja Resumen (marine) debe
        # cuadrar contra el total de Validación restringido a esas mismas
        # pólizas marine (Validación en sí se exporta completa, sin filtrar).
        dif_total_validacion_marine = validacion.loc[validacion['Policy Number'].isin(polizas_marine), 'Diferencia'].sum()
        dif_total_resumen = resumen.loc[resumen['Policy Number'] == 'Total general', 'Suma de Diferencia'].iloc[0]
        if abs(dif_total_validacion_marine - dif_total_resumen) > tolerancia:
            print(f"  Aviso: el total 'Suma de Diferencia' en Resumen (marine, ${dif_total_resumen:,.2f}) no cuadra con "
                  f"el total de Validación restringido a pólizas marine (${dif_total_validacion_marine:,.2f}) — "
                  "revisar _construir_resumen_validacion.")
    except OSError as e:
        raise OSError(f'No se pudo escribir {ruta_salida}. ¿Está abierto en Excel? Ciérralo e intenta de nuevo.') from e


# --- Ejecución: si falta el BDX o Payments de este mes, se avisa y se sigue sin tronar ---
try:
    df_bdx_validacion = _cargar_bdx_net_reserve_validacion(route_account, AñoMes)
    validacion, bdx_detalle_validacion, resumen_zereos_validacion = _construir_validacion(df_update_db, df_bdx_validacion, df_log_zereo_oslr)

    # Pólizas marine = las que existen en Reserve Database (df_update_db).
    # Usado para acotar las hojas Resumen y Outstanding Payments a marine
    # (ver _construir_resumen_validacion / _construir_outstanding_payments_detalle);
    # Validación se sigue exportando completa, sin este filtro.
    polizas_marine = set(df_update_db['INWARD POLICY N°'].astype(str).str.strip().unique())

    # Las hojas Siniestro y Outstanding Payments dependen además de
    # Payments_{AñoMes}.xlsx y de {AñoMes}_BDX_Marine.xlsx (BDX real por claim).
    # Si alguno de los dos no está o le faltan columnas, se exporta igual el
    # resto del reporte con ambas hojas vacías (no se pierde la validación
    # por póliza por un problema en estas fuentes adicionales).
    cols_siniestro_vacio = [
        'Policy Number', 'Claim Number', 'Reserve Database', 'BDX Reserve',
        'Diferencias', 'Outstanding Payments', 'Diferencia por Tipo de Cambio',
    ]
    cols_outstanding_vacio = ['Policy Number', 'Claim Number', 'Reserve Database', 'BDX Reserve', 'Outstanding']
    try:
        df_bdx_claim_validacion = _cargar_bdx_reserve_por_claim_validacion(route_validations, AñoMes)
        df_outstanding_validacion = _cargar_outstanding_payments_validacion(route_account, AñoMes)
        detalle_claims_validacion = _construir_detalle_claims(df_update_db, df_bdx_claim_validacion, df_outstanding_validacion)
        siniestro_detalle_validacion = _construir_siniestro_detalle(detalle_claims_validacion)
        outstanding_detalle_validacion = _construir_outstanding_payments_detalle(detalle_claims_validacion)
    except (FileNotFoundError, KeyError, ValueError) as e:
        print(f'Hojas Siniestro / Outstanding Payments omitidas (se exportan vacías): {e}')
        siniestro_detalle_validacion = pd.DataFrame(columns=cols_siniestro_vacio)
        outstanding_detalle_validacion = pd.DataFrame(columns=cols_outstanding_vacio)
        detalle_claims_validacion = None

    resumen_validacion = _construir_resumen_validacion(validacion, polizas_marine, detalle_claims_validacion)

    ruta_validacion_oslr = f'{route_validations}/Validacion_OSLR_{AñoMes}.xlsx'
    _exportar_validacion(
        validacion, bdx_detalle_validacion, df_log_zereo_oslr, resumen_zereos_validacion,
        siniestro_detalle_validacion, outstanding_detalle_validacion, resumen_validacion,
        polizas_marine, ruta_validacion_oslr, AñoMes,
        df_reconciliacion_AE=df_reconciliacion_AE, df_reconciliacion_CL=df_reconciliacion_CL,
        df_reconciliacion_VA=df_reconciliacion_VA,
    )
except FileNotFoundError as e:
    print(f'Validación OSLR vs BDX omitida: {e}')
except KeyError as e:
    print(f'Validación OSLR vs BDX omitida por columnas faltantes: {e}')

Aviso: 2 filas sin Policy Number en el BDX de 202604 (Net Reserve total 618,994.39). Se agrupan bajo "SIN POLIZA - <Cover>".
Aviso: 3258 filas en 202604_BDX_Marine.xlsx con |Net Reserve (USD)| < 10 se convirtieron a 0 (ruido de punto flotante).
Aviso: 5 Claim Number en 202604_BDX_Marine.xlsx tienen más de una fila. Se suman todas: 5 netean a ~0 (no generan diferencia en Siniestro), 0 NO netean a 0 (reserva real repartida en varias filas — revisar si aplica).
Payments_202604.xlsx: usando solo la hoja 'CL' (Cumulative Claims Paid) para Outstanding Payments — 87 filas.
Aviso: 5 Claim Number no están en Reserve Database ni en el BDX real por claim (archivo BDX Marine); se agregan igual usando su Policy Number de Payments.
Validación OSLR vs BDX exportada: C:/Users/IKAL14/Documents/Integral/Marine/Validaciones/202604/Validacion_OSLR_202604.xlsx
  Pólizas dentro de tolerancia ($1.0): 23
  Con diferencia y CON zereo que podría explicarla: 1
  Con diferencia SIN ningún zereo que la explique (r